# Tool Calling(Function Calling) 과제
---

**학습 목표**

1. Tool Calling이 **왜 필요하고 어떻게 동작하는지** 메시지 흐름 단위로 설명할 수 있다.
2. 파이썬 함수를 **도구 명세(JSON Schema)** 로 정의하고 모델에 전달할 수 있다.
3. 모델이 반환한 `tool_calls`를 실행하고, 결과를 `role="tool"` 메시지로 돌려줄 수 있다.
4. **병렬 호출, 에러 처리, 사람 승인(Human-in-the-loop), 멀티스텝 에이전트 루프** 등 실무 패턴을 구현할 수 있다.

**실습 구성**

| 파트 | 시나리오 | 핵심 개념 |
|---|---|---|
| Part 0 | 개념 이해 & 환경 설정 | Tool Calling 동작 흐름, 메시지 구조 |
| 시나리오 1 | 쇼핑몰 고객상담 챗봇 | 도구 스키마 정의, 호출 → 실행 → 응답 사이클, `tool_choice` |
| 시나리오 2 | 사내 회의실 예약 비서 | 파라미터 추출, Strict 모드, 병렬 호출, 에러 처리, 사람 승인 |
| 시나리오 3 | 매출 분석 리포트 에이전트 | 멀티스텝 에이전트 루프, 도구 체이닝, 안전장치, 실행 추적 |

**사전 준비**: Python 3.9 이상, OpenAI API 키

> 모든 실습은 실제 DB/메일 서버 대신 **가상(Mock) 데이터**를 사용하므로 API 키만 있으면 바로 실행할 수 있습니다.
> 실무에서는 Mock 함수 내부만 실제 DB 조회·API 호출로 바꾸면 됩니다.

---
# Part 0. Tool Calling 개념 이해

## 0-1. Tool Calling이란?

LLM은 학습된 지식으로 **텍스트를 생성**할 뿐, 다음과 같은 일은 스스로 할 수 없습니다.

- 실시간 정보 조회 (오늘 주문 상태, 현재 재고, 환율 등)
- 사내 DB·외부 API 접근
- 정확한 계산, 메일 발송, 예약 등 **실제 행동(Action)**

**Tool Calling**은 모델이 *"이 작업에는 어떤 함수를, 어떤 인자로 호출해야 한다"* 를 **구조화된 JSON**으로 알려주고,
**실제 실행은 개발자의 코드가 담당**하는 방식입니다.

> **가장 중요한 한 가지**
> 모델은 함수를 **직접 실행하지 않습니다.** "이 함수를 이 인자로 불러 주세요"라고 *요청*할 뿐입니다.
> 따라서 **실행 여부·권한·입력 검증의 책임은 전적으로 개발자 코드**에 있습니다.

## 0-2. 동작 흐름

```
  사용자 질문: "ORD-1001 주문 어디쯤 왔어요?"
        │
        ▼
 ① [우리 코드] messages + tools(도구 명세 목록)를 모델에 전달
        │
        ▼
 ② [모델] 도구가 필요한지 스스로 판단
        ├─ 필요 없음 → 바로 텍스트 답변             (finish_reason = "stop")
        └─ 필요함   → tool_calls 반환              (finish_reason = "tool_calls")
                       예) get_order_status(order_id="ORD-1001")
        │
        ▼
 ③ [우리 코드] 실제 함수 실행 → 결과를 role="tool" 메시지로 대화에 추가
        │
        ▼
 ④ [모델] 도구 결과를 읽고 최종 답변 생성
          (추가 정보가 필요하면 다시 tool_calls → ②로 반복)
```

## 0-3. 핵심 용어

| 용어 | 설명 |
|---|---|
| `tools` | 모델에게 알려주는 **사용 가능한 도구 목록**. 각 도구는 이름·설명·파라미터 스키마로 구성 |
| `name` | 함수 이름. 모델이 호출할 때 이 이름을 사용 |
| `description` | **언제, 왜** 이 도구를 써야 하는지에 대한 설명. 모델의 도구 선택 정확도를 좌우하는 가장 중요한 요소 |
| `parameters` | 함수 인자를 **JSON Schema** 형식으로 정의 (`type`, `properties`, `required`, `enum` 등) |
| `tool_calls` | 모델 응답에 포함되는 **호출 요청 목록**. 한 번에 여러 개가 올 수 있음(병렬 호출) |
| `tool_call_id` | 각 호출 요청의 고유 ID. 결과를 돌려줄 때 **반드시 같은 ID로 짝**을 맞춰야 함 |
| `role: "tool"` | 함수 실행 결과를 모델에게 돌려주는 메시지 역할 |
| `finish_reason` | 응답 종료 이유. `"tool_calls"`면 도구 실행이 필요하다는 뜻 |
| `tool_choice` | 도구 사용 정책 (`auto` / `none` / `required` / 특정 함수 강제) |

## 0-4. 메시지 구조 미리보기

도구 호출 한 사이클이 끝나면 `messages` 리스트는 다음과 같은 모양이 됩니다.

```json
[
  {"role": "system",    "content": "당신은 쇼핑몰 상담원입니다."},
  {"role": "user",      "content": "ORD-1001 어디쯤 왔어요?"},
  {"role": "assistant", "content": null,
   "tool_calls": [{"id": "call_abc", "type": "function",
                   "function": {"name": "get_order_status",
                                "arguments": "{\"order_id\": \"ORD-1001\"}"}}]},
  {"role": "tool", "tool_call_id": "call_abc",
   "content": "{\"status\": \"배송중\", \"expected_delivery\": \"2026-10-09\"}"},
  {"role": "assistant", "content": "고객님의 주문은 현재 배송 중이며 10월 9일 도착 예정입니다."}
]
```

> `arguments`는 **딕셔너리가 아니라 JSON 문자열**입니다. 반드시 `json.loads()`로 파싱해야 합니다.

## 0-5. 환경 설정

패키지를 설치하고 OpenAI 클라이언트를 준비합니다.

In [1]:
%pip install -q openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

# 1) .env 파일에 OPENAI_API_KEY=sk-... 형태로 저장해 두었다면 자동으로 읽어옵니다.
load_dotenv()

# 2) 환경변수에 키가 없으면 직접 입력받습니다. (입력값은 화면에 표시되지 않음)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key를 입력하세요: ")

client = OpenAI()

# Tool Calling을 지원하는 모델이면 무엇이든 사용 가능합니다. (예: gpt-4o-mini, gpt-4.1-mini, gpt-4o ...)
MODEL = "gpt-4o-mini"


def pretty(obj):
    """한글이 깨지지 않도록 JSON을 보기 좋게 출력하는 헬퍼"""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump(exclude_none=True)
    print(json.dumps(obj, ensure_ascii=False, indent=2))


print("설정 완료! 사용 모델:", MODEL)

설정 완료! 사용 모델: gpt-4o-mini


In [3]:
# 연결 테스트 (도구 없이 일반 대화)
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "한 문장으로 자기소개 해줘."}],
)
print(resp.choices[0].message.content)

안녕하세요, 저는 여러분과 소통하고 지식을 나누는 것을 좋아하는 AI 언어 모델입니다.


---
# 시나리오 1. 쇼핑몰 고객상담 챗봇 — 주문 조회 & 반품 정책 안내

## 배경

온라인 쇼핑몰 **'캠퍼스몰'** 고객센터에는 *"제 주문 어디쯤 왔어요?"*, *"이거 반품 되나요?"* 같은 문의가 하루 수백 건씩 들어옵니다.
상담원은 매번 주문 시스템을 열어 조회하고 반품 규정을 찾아 답변합니다.

본 과제에서는 **주문 DB 조회**와 **반품 정책 조회** 두 가지 도구를 가진 상담 챗봇을 만들며
Tool Calling의 **기본 사이클 전체를 한 단계씩** 직접 따라가 봅니다.

## 학습 포인트

- 파이썬 함수를 **도구 명세(JSON Schema)** 로 변환하는 방법
- 모델 응답에서 `tool_calls`를 꺼내 **직접 실행하고 결과를 돌려주는** 과정
- 도구가 필요 없는 질문에는 모델이 **도구 없이 바로 답한다**는 점
- `tool_choice` 옵션으로 도구 사용 정책을 제어하는 방법

## 1-1. 백엔드 준비: 가상 데이터와 실제 함수

실무라면 주문 DB나 ERP API를 조회하겠지만, 여기서는 딕셔너리로 대신합니다.
**도구 함수는 평범한 파이썬 함수**라는 점에 주목하세요. LLM과 관련된 코드는 하나도 없습니다.

In [65]:
# ---- 가상 주문 DB ----
ORDERS = {
    "ORD-1001": {
        "customer": "김민수", "product": "무선 이어폰 Pro", "category": "전자기기",
        "status": "배송중", "order_date": "2026-10-05",
        "courier": "한진택배", "tracking_no": "5012-3344-7788",
        "expected_delivery": "2026-10-09",
    },
    "ORD-1002": {
        "customer": "이지은", "product": "린넨 셔츠 (M)", "category": "의류",
        "status": "배송완료", "order_date": "2026-09-28",
        "courier": "CJ대한통운", "tracking_no": "6201-1100-2233",
        "delivered_date": "2026-10-01",
    },
    "ORD-1003": {
        "customer": "박서준", "product": "유기농 견과류 세트", "category": "식품",
        "status": "결제완료", "order_date": "2026-10-07",
        "expected_delivery": "2026-10-10",
    },
}

# ---- 카테고리별 반품 정책 ----
RETURN_POLICIES = {
    "전자기기": {"period_days": 7,  "condition": "미개봉 상태에 한해 가능",
                 "shipping_fee": "단순변심 시 왕복 배송비 6,000원 고객 부담"},
    "의류":     {"period_days": 14, "condition": "택 제거 전, 착용 흔적이 없을 것",
                 "shipping_fee": "단순변심 시 왕복 배송비 5,000원 고객 부담"},
    "식품":     {"period_days": 0,  "condition": "단순변심 반품 불가 (상품 하자 시 수령 후 24시간 이내 접수)",
                 "shipping_fee": "상품 하자 시 무료"},
}


def get_order_status(order_id: str) -> dict:
    """주문번호로 주문 상태와 배송 정보를 조회한다."""
    order_id = order_id.strip().upper()
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "message": f"주문번호 {order_id}에 해당하는 주문이 없습니다."}
    return {"found": True, "order_id": order_id, **order}


def get_return_policy(category: str) -> dict:
    """정책을 조회한다."""
    policy = RETURN_POLICIES.get(category)
    if policy is None:
        return {"found": False, "message": f"'{category}' 카테고리의 정책이 없습니다."}
    return {"found": True, "category": category, **policy}


# 함수 단독 테스트 (LLM 없이도 잘 동작해야 함)
pretty(get_order_status("ord-1001"))
pretty(get_return_policy("의류"))

{
  "found": true,
  "order_id": "ORD-1001",
  "customer": "김민수",
  "product": "무선 이어폰 Pro",
  "category": "전자기기",
  "status": "배송중",
  "order_date": "2026-10-05",
  "courier": "한진택배",
  "tracking_no": "5012-3344-7788",
  "expected_delivery": "2026-10-09"
}
{
  "found": true,
  "category": "의류",
  "period_days": 14,
  "condition": "택 제거 전, 착용 흔적이 없을 것",
  "shipping_fee": "단순변심 시 왕복 배송비 5,000원 고객 부담"
}


## 1-2. 도구 명세(Tool Schema) 작성

모델은 우리 파이썬 코드를 볼 수 없습니다. 오직 **도구 명세만 보고** 어떤 도구를 언제 쓸지 판단합니다.
따라서 명세는 *"모델을 위한 API 문서"* 라고 생각하고 작성해야 합니다.

**좋은 description 작성 요령**

| 항목 | 나쁜 예 | 좋은 예 |
|---|---|---|
| 도구 설명 | "주문 조회" | "주문번호로 주문 상태·택배사·송장번호·도착 예정일을 조회한다. 고객이 배송 현황이나 주문 상태를 물을 때 사용." |
| 파라미터 설명 | "id" | "주문번호. 'ORD-' 로 시작하는 형식 (예: ORD-1001)" |
| 값의 범위 | 자유 문자열 | `enum`으로 허용값 고정 → 모델이 엉뚱한 값을 만들지 않음 |

In [34]:
cs_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": (
                "주문번호로 주문 상태, 상품명, 카테고리, 택배사, 송장번호, 도착 예정일을 조회한다. "
                "고객이 배송 현황, 주문 상태, 도착 시점을 물어볼 때 사용한다."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "주문번호. 'ORD-'로 시작하는 형식 (예: ORD-1001)",
                    }
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_return_policy",
            "description": (
                "정책을 조회한다. "
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "enum": ["전자기기", "의류", "식품"],
                        "description": "상품 카테고리",
                    }
                },
                "required": ["category"],
            },
        },
    },
]

CS_SYSTEM_PROMPT = """당신은 온라인 쇼핑몰 '캠퍼스몰'의 친절한 고객상담 챗봇입니다.
- 주문/배송/반품 관련 정보는 반드시 제공된 도구로 조회한 사실만 답변하세요. 추측하지 마세요.
- 고객센터 운영시간: 평일 09:00~18:00 (점심 12:00~13:00), 주말·공휴일 휴무
- 답변은 간결하고 공손하게 3~4문장 이내로 작성하세요."""

## 1-3. [Step 1] 모델에 질문 보내기 — 응답 구조 관찰

먼저 도구 명세와 함께 질문을 보내고, 모델이 **무엇을 돌려주는지** 자세히 살펴봅니다.

In [17]:
messages = [
    {"role": "system", "content": CS_SYSTEM_PROMPT},
    {"role": "user", "content": "안녕하세요, 주문번호 ORD-1001 지금 어디쯤 왔나요?"},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=cs_tools,          # <-- 사용 가능한 도구 목록 전달
)

assistant_msg = response.choices[0].message

print("finish_reason :", response.choices[0].finish_reason)
print("content       :", assistant_msg.content)
print("tool_calls 수 :", len(assistant_msg.tool_calls or []))
print()
for tc in assistant_msg.tool_calls or []:
    print("  id        :", tc.id)
    print("  name      :", tc.function.name)
    print("  arguments :", tc.function.arguments, "  <-- type:", type(tc.function.arguments).__name__)

finish_reason : tool_calls
content       : None
tool_calls 수 : 1

  id        : call_tL4o2B7fVdN5C2K5Ec6yRgrg
  name      : get_order_status
  arguments : {"order_id":"ORD-1001"}   <-- type: str


**관찰 포인트**

- `finish_reason`이 `"tool_calls"` → 모델이 *"답하기 전에 도구 실행이 필요하다"* 고 판단했습니다.
- `content`는 `None` → 아직 사용자에게 줄 답변이 없습니다.
- `arguments`는 **문자열(str)** 입니다. 실행하려면 `json.loads()`로 딕셔너리로 바꿔야 합니다.
- 모델은 사용자 문장 *"주문번호 ORD-1001 지금 어디쯤..."* 에서 `order_id="ORD-1001"`을 **스스로 추출**했습니다.

## 1-4. [Step 2] 함수 실행 & 결과를 대화에 추가

이제 **우리 코드가** 실제 함수를 실행합니다. 순서는 다음과 같습니다.

1. 모델의 응답(assistant 메시지, `tool_calls` 포함)을 **먼저** `messages`에 추가
2. 각 `tool_call`마다 함수 실행
3. 결과를 `{"role": "tool", "tool_call_id": ..., "content": ...}` 형태로 추가

> 함수 이름(문자열) → 실제 함수 객체를 찾기 위해 **도구 레지스트리(dict)** 를 사용합니다.
> `eval()`이나 `globals()`로 함수를 찾으면 모델이 임의의 함수를 실행하게 될 수 있으니 **절대 사용하지 마세요.**

In [18]:
# 함수 이름 → 실제 함수 매핑 (허용된 함수만 등록 = 화이트리스트)
CS_TOOL_REGISTRY = {
    "get_order_status": get_order_status,
    "get_return_policy": get_return_policy,
}

# (1) 모델의 tool_calls 응답을 대화 기록에 먼저 추가
messages.append({
    "role": "assistant",
    "content": assistant_msg.content,
    "tool_calls": [tc.model_dump() for tc in assistant_msg.tool_calls],
})

# (2)(3) 각 도구 호출을 실행하고 결과 추가
for tc in assistant_msg.tool_calls:
    func = CS_TOOL_REGISTRY[tc.function.name]
    args = json.loads(tc.function.arguments)         # 문자열 -> dict
    result = func(**args)                            # 실제 함수 실행!

    print(f"실행: {tc.function.name}({args})")
    pretty(result)

    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,                       # 어떤 요청에 대한 결과인지 짝 맞추기
        "content": json.dumps(result, ensure_ascii=False),  # 결과는 문자열로 전달
    })

print("\n현재 대화 기록의 role 순서:", [m["role"] for m in messages])

실행: get_order_status({'order_id': 'ORD-1001'})
{
  "found": true,
  "order_id": "ORD-1001",
  "customer": "김민수",
  "product": "무선 이어폰 Pro",
  "category": "전자기기",
  "status": "배송중",
  "order_date": "2026-10-05",
  "courier": "한진택배",
  "tracking_no": "5012-3344-7788",
  "expected_delivery": "2026-10-09"
}

현재 대화 기록의 role 순서: ['system', 'user', 'assistant', 'tool']


## 1-5. [Step 3] 도구 결과를 바탕으로 최종 답변 받기

도구 결과가 포함된 `messages`를 다시 모델에 보내면, 모델이 결과를 읽고 자연어 답변을 만듭니다.

In [19]:
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=cs_tools,
)

print("finish_reason:", final_response.choices[0].finish_reason)
print("-" * 60)
print(final_response.choices[0].message.content)

finish_reason: stop
------------------------------------------------------------
안녕하세요, 김민수님. 주문번호 ORD-1001의 '무선 이어폰 Pro'는 현재 배송 중입니다. 택배사는 한진택배이며, 송장번호는 5012-3344-7788입니다. 예상 도착일은 2026년 10월 9일입니다.


**정리: 하나의 질문에 API 호출이 2번** 일어났습니다.

| 호출 | 입력 | 출력 |
|---|---|---|
| 1차 | system + user | `tool_calls` (get_order_status 요청) |
| (우리 코드) | `tool_calls` | 함수 실행 결과 → `role="tool"` 메시지 |
| 2차 | system + user + assistant(tool_calls) + tool(결과) | 최종 자연어 답변 |

이 *"요청 → 실행 → 결과 전달 → 답변"* 사이클이 Tool Calling의 전부입니다. 이후 실습은 모두 이 사이클의 응용입니다.

## 1-6. 재사용 가능한 함수로 묶기

실제 서비스에서는 질문에 따라 **도구가 필요 없을 수도**, **여러 번 필요할 수도** 있습니다.
예) *"셔츠 반품 되나요?"* → 주문 조회로 카테고리 확인 → 반품 정책 조회 (2단계)

그래서 `tool_calls`가 없을 때까지 사이클을 반복하는 함수로 만듭니다. (무한 루프 방지를 위해 최대 횟수 제한)

In [20]:
def ask_cs_bot(question: str, max_rounds: int = 5, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": CS_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for round_no in range(1, max_rounds + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=cs_tools)
        msg = response.choices[0].message

        # 도구 호출이 없으면 = 최종 답변
        if not msg.tool_calls:
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = CS_TOOL_REGISTRY[tc.function.name](**args)
            if verbose:
                print(f"  [라운드 {round_no}] {tc.function.name}({args})")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "죄송합니다. 요청을 처리하지 못했습니다. 상담원에게 연결해 드리겠습니다."

In [29]:
test_questions = [
    "ORD-1002로 셔츠 샀는데 사이즈가 안 맞아요. 반품 가능한가요?",  # 도구 2단계 연쇄 호출
    "고객센터 운영시간이 어떻게 되나요?",                         # 도구 불필요 (시스템 프롬프트에 정보 있음)
    "ORD-9999 배송 조회 부탁드려요.",                           # 존재하지 않는 주문
    "식품도 단순변심으로 반품되나요?",                            # 반품 정책만 조회
]

for q in test_questions:
    print("=" * 70)
    print("고객:", q)
    answer = ask_cs_bot(q)
    print("챗봇:", answer)

print("=" * 70)
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": CS_SYSTEM_PROMPT},
              {"role": "user", "content": "식품도 단순변심으로 반품되나요?"}],
    tools=cs_tools,
)
print(r.choices[0].finish_reason, r.choices[0].message.tool_calls)

고객: ORD-1002로 셔츠 샀는데 사이즈가 안 맞아요. 반품 가능한가요?
  [라운드 1] get_order_status({'order_id': 'ORD-1002'})
  [라운드 2] get_return_policy({'category': '의류'})
챗봇: 주문하신 린넨 셔츠는 반품 가능합니다. 단, 반품을 원하시면 14일 이내에 신청하셔야 하며, 택을 제거하지 않고 착용 흔적이 없어야 합니다. 또한, 단순 변심 시 왕복 배송비 5,000원이 고객님 부담입니다. 추가로 궁금한 점이 있으시면 말씀해 주세요!
고객: 고객센터 운영시간이 어떻게 되나요?
챗봇: 고객센터 운영시간은 평일 09:00부터 18:00까지이며, 점심시간은 12:00부터 13:00까지입니다. 주말과 공휴일은 휴무입니다. 추가로 궁금하신 점이 있으시면 언제든지 문의해 주세요!
고객: ORD-9999 배송 조회 부탁드려요.
  [라운드 1] get_order_status({'order_id': 'ORD-9999'})
챗봇: 죄송하지만, 주문번호 ORD-9999에 해당하는 주문이 없습니다. 주문번호를 다시 확인해 주시거나, 다른 문의 사항이 있으시면 알려주세요.
고객: 식품도 단순변심으로 반품되나요?
  [라운드 1] get_return_policy({'category': '식품'})
챗봇: 식품은 단순변심으로 반품이 불가합니다. 상품 하자가 있을 경우, 수령 후 24시간 이내에 접수하셔야 반품이 가능합니다. 이 경우 반품 배송비는 무료입니다. 추가 문의 사항이 있으시면 말씀해 주세요.
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_yU2UG9Fk4Uuw11DElfFsSEFF', function=Function(arguments='{"category":"식품"}', name='get_return_policy'), type='function')]


**관찰 포인트**

- 운영시간 질문에는 **도구를 호출하지 않고** 바로 답합니다. → 모델이 도구 사용 여부를 스스로 판단
- 셔츠 반품 질문은 **주문 조회 → (카테고리=의류 확인) → 반품 정책 조회** 순서로 2라운드가 진행됩니다.
- 없는 주문번호는 함수가 `found: False`를 돌려주고, 모델이 이를 자연스럽게 안내합니다.
  → **도구 결과에 "실패 이유"를 명확히 담아 주는 것**이 좋은 답변의 비결입니다.

## 1-7. `tool_choice`로 도구 사용 정책 제어하기

| 값 | 동작 | 실무 활용 예 |
|---|---|---|
| `"auto"` (기본값) | 모델이 도구 사용 여부를 스스로 결정 | 일반 챗봇 |
| `"none"` | 도구를 절대 사용하지 않음 | 요약·인사 등 도구가 불필요한 단계 |
| `"required"` | 반드시 하나 이상의 도구를 호출 | 분류기, 데이터 추출 파이프라인 |
| `{"type": "function", "function": {"name": "..."}}` | 지정한 도구를 강제로 호출 | 특정 양식으로 정보 추출 강제 |

같은 질문 *"안녕하세요!"* 를 4가지 정책으로 보내 비교해 봅시다.

In [32]:
choices = {
    "auto": "auto",
    "none": "none",
    "required": "required",
    "강제(get_return_policy)": {"type": "function", "function": {"name": "get_return_policy"}},
}

for label, choice in choices.items():
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": CS_SYSTEM_PROMPT},
                  {"role": "user", "content": "안녕하세요!"}],
        tools=cs_tools,
        tool_choice=choice,
    )
    m = r.choices[0].message
    if m.tool_calls:
        calls = [f"{tc.function.name}({tc.function.arguments})" for tc in m.tool_calls]
        print(f"[{label:<22}] 도구 호출 → {calls}")
    else:
        print(f"[{label:<22}] 텍스트 답변 → {m.content[:50]}")

[auto                  ] 텍스트 답변 → 안녕하세요! 캠퍼스몰 고객상담 챗봇입니다. 어떻게 도와드릴까요?
[none                  ] 텍스트 답변 → 안녕하세요! 캠퍼스몰에 오신 것을 환영합니다. 무엇을 도와드릴까요?
[required              ] 도구 호출 → ['get_order_status({"order_id":"ORD-1001"})']
[강제(get_return_policy) ] 도구 호출 → ['get_return_policy({"category":"의류"})']


> `required`나 강제 지정은 인사말처럼 도구가 필요 없는 상황에서도 **억지로 도구를 호출**하게 만듭니다.
> 일반 대화형 서비스는 `auto`를 기본으로 쓰고, 정해진 데이터 추출 작업에서만 강제 옵션을 쓰세요.

## 1-8. 연습 문제

**[문제 1]** 주문 취소 도구 `cancel_order(order_id, reason)`를 추가해 보세요.
- 규칙: `status`가 `"결제완료"`인 주문만 취소 가능, 그 외에는 취소 불가 사유를 반환
- `cs_tools`, `CS_TOOL_REGISTRY`에 등록하고 *"ORD-1003 취소해 주세요. 단순변심이에요."* / *"ORD-1001 취소할래요"* 로 테스트

**[문제 2]** `get_return_policy`의 description을 `"정책 조회"`로 짧게 바꾼 뒤, 셔츠 반품 질문의 결과가 어떻게 달라지는지 관찰해 보세요.

아래는 문제 1의 예시 답안입니다. 먼저 직접 작성해 본 뒤 비교해 보세요.

In [72]:
# [문제1] 작성.
#예시답안의 tools 등록과 registry에서 제거
CS_TOOL_REGISTRY.pop("cancel_order", None)

cs_tools[:] = [t for t in cs_tools if t["function"]["name"] != "cancel_order"]

print(f"cs_tools에 남은 tool: {[t['function']['name'] for t in cs_tools]}")
print(f":CS_TOOL_REGISTRY_등록된_tools: {CS_TOOL_REGISTRY}")
print('\n','='*70)
# 문제 1번 함수 정의
def cancel_order(order_id:str, reason:str):
    """주문번호에 해당하는 주문을 파악해서 주문 취소 가능 여부 판단한다. 결제완료일 때 취소를 진행하고 사유를 기록한다."""
    order_id = order_id.strip().upper()
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False,"message": f"주문번호{order_id}는 존재하지 않는 주문번호입니다."}
    if order["status"] == "결제완료":
        order["status"] = "취소완료"
        order["cancel_reason"] = reason
        return {"found": True,"order_id":order_id,"message":"주문이 취소가 완료되었습니다. 환불은 영업일 기준 3일 내에 처리됩니다."}
    if order["status"] !="결제완료":
        return {"found":True, "message": f"고객님 {order['status']} 상태의 주문은 취소가 불가합니다. 결제완료 상태의 주문만 취소 가능합니다."}

print(cancel_order(order_id= "ord-1239", reason="그냥 취고 하고 싶은데둉"))
print('\n','='*70)
# cs_tools 및 SC_TOOL_REGISTRY에 등록.
cancel_tool = {
    "type": "function",
    "function": {
        "name": "cancel_order",
        "description": "주문을 취소한다. 결제완료 상태의 주문만 취소 가능하다. 고객이 주문 취소를 명확히 요청한 경우에만 사용한다. 고객이 취소 사유를 밝히지 않아도 정책상 가능하면 취소한다. ",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "주문번호 (예: ORD-1003)"},
                "reason": {"type": "string", "description": "고객이 밝힌 취소 사유. 언급이 없으면 '미기재'"},
            },
            "required": ["order_id", "reason"],
        },
    },
}
if not any(t["function"]["name"] == "cancel_order" for t in cs_tools):
    cs_tools.append(cancel_tool)
CS_TOOL_REGISTRY["cancel_order"] = cancel_order
print("cancel_tool_등록완료")
print(cs_tools)
print(CS_TOOL_REGISTRY)
print('\n','='*70)

for q in ["ORD-1003 취소해 주세요. 단순변심이에요.", "ORD-1001 취소할래요.","셔츠반품"]:
    print("고객:", q)
    print("챗봇:", ask_cs_bot(q), "\n")

cs_tools에 남은 tool: ['get_order_status', 'get_return_policy']
:CS_TOOL_REGISTRY_등록된_tools: {'get_order_status': <function get_order_status at 0x00000160BE0A1BC0>, 'get_return_policy': <function get_return_policy at 0x00000160BE0A20C0>}

{'found': False, 'message': '주문번호ORD-1239는 존재하지 않는 주문번호입니다.'}

cancel_tool_등록완료
[{'type': 'function', 'function': {'name': 'get_order_status', 'description': '주문번호로 주문 상태, 상품명, 카테고리, 택배사, 송장번호, 도착 예정일을 조회한다. 고객이 배송 현황, 주문 상태, 도착 시점을 물어볼 때 사용한다.', 'parameters': {'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': "주문번호. 'ORD-'로 시작하는 형식 (예: ORD-1001)"}}, 'required': ['order_id']}}}, {'type': 'function', 'function': {'name': 'get_return_policy', 'description': '정책을 조회한다. ', 'parameters': {'type': 'object', 'properties': {'category': {'type': 'string', 'enum': ['전자기기', '의류', '식품'], 'description': '상품 카테고리'}}, 'required': ['category']}}}, {'type': 'function', 'function': {'name': 'cancel_order', 'description': '주문을 취소한다. 결제완료 상태의 주문

In [70]:
# ---- [문제 1] 예시 답안 ----
def cancel_order(order_id: str, reason: str) -> dict:
    order_id = order_id.strip().upper()
    order = ORDERS.get(order_id)
    if order is None:
        return {"success": False, "message": f"주문번호 {order_id}를 찾을 수 없습니다."}
    if order["status"] != "결제완료":
        return {"success": False,
                "message": f"현재 상태가 '{order['status']}'이므로 취소할 수 없습니다. 출고 전(결제완료) 주문만 취소 가능합니다."}
    order["status"] = "취소완료"
    order["cancel_reason"] = reason
    return {"success": True, "order_id": order_id, "message": "주문이 취소되었습니다. 환불은 3영업일 내 처리됩니다."}


cancel_tool = {
    "type": "function",
    "function": {
        "name": "cancel_order",
        "description": "주문을 취소한다. 출고 전(결제완료) 상태의 주문만 취소 가능하다. 고객이 주문 취소를 명확히 요청한 경우에만 사용한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "주문번호 (예: ORD-1003)"},
                "reason": {"type": "string", "description": "고객이 밝힌 취소 사유. 언급이 없으면 '미기재'"},
            },
            "required": ["order_id", "reason"],
        },
    },
}

if not any(t["function"]["name"] == "cancel_order" for t in cs_tools):
    cs_tools.append(cancel_tool)
CS_TOOL_REGISTRY["cancel_order"] = cancel_order

for q in ["ORD-1003 취소해 주세요. 단순변심이에요.", "ORD-1001 취소할래요."]:
    print("고객:", q)
    print("챗봇:", ask_cs_bot(q), "\n")

고객: ORD-1003 취소해 주세요. 단순변심이에요.
  [라운드 1] cancel_order({'order_id': 'ORD-1003', 'reason': '단순변심'})
챗봇: 죄송하지만, 주문번호 ORD-1003은 현재 '취소완료' 상태로 취소가 불가능합니다. 주문이 결제완료 상태여야 취소가 가능합니다. 추가적인 문의 사항이 있으시면 말씀해 주세요. 

고객: ORD-1001 취소할래요.
  [라운드 1] cancel_order({'order_id': 'ORD-1001', 'reason': '미기재'})
챗봇: 주문번호 ORD-1001은 현재 '배송중' 상태로, 출고 전의 결제 완료 주문만 취소가 가능합니다. 다른 문의사항이 있으시면 말씀해 주세요. 



---
# 시나리오 2. 사내 회의실 예약 비서 — 파라미터 추출 · 병렬 호출 · 에러 처리 · 사람 승인

## 배경

사내 메신저에 회의실 예약 비서를 붙이려고 합니다. 직원들은 이렇게 말합니다.

> *"내일 오후 3시에 6명 회의할 방 잡아줘"*
> *"다음주 월요일 10시 반부터 한 시간, 화상회의 되는 곳 있어?"*

시나리오 1과 달리 이번에는 **"쓰기(Write) 작업"** 인 예약이 포함됩니다.
조회는 틀려도 다시 물으면 되지만, **예약·취소·결제는 잘못 실행되면 되돌리기 어렵습니다.**
따라서 *정확한 파라미터 추출*, *입력 검증*, *에러 처리*, *사람의 최종 승인*이 중요합니다.

## 학습 포인트

1. "내일", "오후 3시", "한 시간" 같은 자연어를 **정확한 구조화 파라미터**로 추출하기 (오늘 날짜 주입)
2. **Strict 모드**로 모델이 스키마를 100% 준수하도록 강제하기
3. 한 번의 응답에 여러 도구 호출이 담기는 **병렬 호출(Parallel Tool Calls)** 처리
4. 함수 에러를 프로그램 중단 대신 **결과로 돌려줘서 모델이 스스로 복구**하게 하기
5. 쓰기 작업 전 **사람 승인(Human-in-the-loop)** 단계 넣기

## 2-1. 백엔드 준비: 회의실·예약 데이터와 함수

실습 재현성을 위해 **오늘 날짜를 2026-10-08(목)** 로 고정합니다.

In [ ]:
from datetime import datetime, timedelta
import re
import copy

TODAY = "2026-10-08"   # 실습용 고정 날짜 (실무에서는 datetime.now() 사용)

ROOMS = {
    "R101": {"name": "소회의실 A", "capacity": 4,  "floor": 1, "equipment": ["TV"]},
    "R102": {"name": "소회의실 B", "capacity": 6,  "floor": 1, "equipment": ["화이트보드"]},
    "R201": {"name": "중회의실",   "capacity": 10, "floor": 2, "equipment": ["빔프로젝터", "화상회의"]},
    "R301": {"name": "대회의실",   "capacity": 20, "floor": 3, "equipment": ["빔프로젝터", "화상회의", "마이크"]},
}

INITIAL_RESERVATIONS = [
    {"id": "RSV-001", "room_id": "R102", "date": "2026-10-09", "start_time": "14:00", "end_time": "16:00", "title": "마케팅 주간회의"},
    {"id": "RSV-002", "room_id": "R201", "date": "2026-10-09", "start_time": "15:00", "end_time": "17:00", "title": "신규 프로젝트 킥오프"},
    {"id": "RSV-003", "room_id": "R101", "date": "2026-10-12", "start_time": "10:00", "end_time": "11:00", "title": "1on1 면담"},
]
RESERVATIONS = copy.deepcopy(INITIAL_RESERVATIONS)


# ---------- 입력 검증 헬퍼 ----------
def _validate_date(date: str):
    try:
        d = datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"날짜 형식이 잘못되었습니다: '{date}' (YYYY-MM-DD 형식 필요)")
    if d < datetime.strptime(TODAY, "%Y-%m-%d"):
        raise ValueError(f"지난 날짜({date})는 예약할 수 없습니다. 오늘은 {TODAY}입니다.")
    if d.weekday() >= 5:
        raise ValueError(f"{date}은(는) 주말입니다. 회의실은 평일에만 운영됩니다.")


def _validate_time_range(start_time: str, end_time: str):
    for t in (start_time, end_time):
        if not re.fullmatch(r"([01]\d|2[0-3]):[0-5]\d", t):
            raise ValueError(f"시간 형식이 잘못되었습니다: '{t}' (HH:MM 24시간 형식 필요)")
    if start_time >= end_time:
        raise ValueError(f"종료 시간({end_time})이 시작 시간({start_time})보다 늦어야 합니다.")
    if start_time < "08:00" or end_time > "21:00":
        raise ValueError("회의실 운영 시간은 08:00~21:00입니다.")


def _is_overlap(s1, e1, s2, e2) -> bool:
    return s1 < e2 and s2 < e1


# ---------- 도구 함수 ----------
def check_room_availability(date: str, start_time: str, end_time: str, min_capacity=None) -> dict:
    """특정 날짜·시간대에 예약 가능한 회의실 목록을 반환한다."""
    _validate_date(date)
    _validate_time_range(start_time, end_time)

    available = []
    for room_id, room in ROOMS.items():
        if min_capacity and room["capacity"] < min_capacity:
            continue
        busy = any(
            r["room_id"] == room_id and r["date"] == date
            and _is_overlap(start_time, end_time, r["start_time"], r["end_time"])
            for r in RESERVATIONS
        )
        if not busy:
            available.append({"room_id": room_id, **room})

    return {"date": date, "time": f"{start_time}~{end_time}",
            "available_count": len(available), "available_rooms": available}


def book_room(room_id: str, date: str, start_time: str, end_time: str, title: str) -> dict:
    """회의실을 예약한다. (쓰기 작업)"""
    if room_id not in ROOMS:
        raise ValueError(f"존재하지 않는 회의실입니다: {room_id}. 가능한 ID: {list(ROOMS)}")
    _validate_date(date)
    _validate_time_range(start_time, end_time)

    for r in RESERVATIONS:
        if r["room_id"] == room_id and r["date"] == date and \
           _is_overlap(start_time, end_time, r["start_time"], r["end_time"]):
            raise ValueError(
                f"{ROOMS[room_id]['name']}({room_id})은(는) {date} {r['start_time']}~{r['end_time']}에 "
                f"'{r['title']}' 회의로 이미 예약되어 있습니다."
            )

    new_id = f"RSV-{len(RESERVATIONS) + 1:03d}"
    RESERVATIONS.append({"id": new_id, "room_id": room_id, "date": date,
                         "start_time": start_time, "end_time": end_time, "title": title})
    return {"success": True, "reservation_id": new_id, "room": ROOMS[room_id]["name"],
            "date": date, "time": f"{start_time}~{end_time}", "title": title}


# 함수 단독 테스트
pretty(check_room_availability("2026-10-09", "15:00", "16:00", 6))
try:
    book_room("R201", "2026-10-09", "15:30", "16:30", "테스트")
except ValueError as e:
    print("예상된 에러:", e)

{
  "date": "2026-10-09",
  "time": "15:00~16:00",
  "available_count": 1,
  "available_rooms": [
    {
      "room_id": "R301",
      "name": "대회의실",
      "capacity": 20,
      "floor": 3,
      "equipment": [
        "빔프로젝터",
        "화상회의",
        "마이크"
      ]
    }
  ]
}
예상된 에러: 중회의실(R201)은(는) 2026-10-09 15:00~17:00에 '신규 프로젝트 킥오프' 회의로 이미 예약되어 있습니다.


## 2-2. Strict 모드로 도구 명세 작성

기본 모드에서도 모델은 대부분 스키마를 잘 따르지만, 가끔 **필수 인자를 빠뜨리거나 정의되지 않은 인자를 추가**할 수 있습니다.
`"strict": true`를 지정하면 모델 출력이 **스키마를 반드시 준수**하도록 보장됩니다(Structured Outputs).

**Strict 모드 규칙**

| 규칙 | 설명 |
|---|---|
| `"additionalProperties": false` | 모든 object에 필수 |
| 모든 속성을 `required`에 나열 | 선택(optional) 인자도 required에 넣어야 함 |
| 선택 인자 표현 | `"type": ["integer", "null"]` 처럼 **null 허용**으로 표현 |

In [75]:
room_tools = [
    {
        "type": "function",
        "function": {
            "name": "check_room_availability",
            "description": (
                "지정한 날짜와 시간대에 예약 가능한 회의실 목록(이름, 수용인원, 층, 장비)을 조회한다. "
                "예약 전에는 반드시 이 도구로 가용 여부를 먼저 확인한다."
            ),
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "date":       {"type": "string", "description": "날짜, YYYY-MM-DD 형식"},
                    "start_time": {"type": "string", "description": "시작 시간, 24시간 HH:MM 형식 (예: 15:00)"},
                    "end_time":   {"type": "string", "description": "종료 시간, 24시간 HH:MM 형식. 언급이 없으면 시작 후 1시간"},
                    "min_capacity": {
                        "type": ["integer", "null"],
                        "description": "필요한 최소 수용 인원. 인원 언급이 없으면 null",
                    },
                },
                "required": ["date", "start_time", "end_time", "min_capacity"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "book_room",
            "description": (
                "회의실을 예약한다. 되돌리기 어려운 쓰기 작업이므로, 사용자가 특정 회의실 예약을 "
                "명확히 요청했을 때만 호출한다."
            ),
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "room_id":    {"type": "string", "enum": list(ROOMS.keys()), "description": "회의실 ID"},
                    "date":       {"type": "string", "description": "날짜, YYYY-MM-DD 형식"},
                    "start_time": {"type": "string", "description": "시작 시간, HH:MM"},
                    "end_time":   {"type": "string", "description": "종료 시간, HH:MM"},
                    "title":      {"type": "string", "description": "회의 제목. 언급이 없으면 '회의'"},
                },
                "required": ["room_id", "date", "start_time", "end_time", "title"],
                "additionalProperties": False,
            },
        },
    },
]

ROOM_TOOL_REGISTRY = {
    "check_room_availability": check_room_availability,
    "book_room": book_room,
}

## 2-3. 시스템 프롬프트에 "오늘 날짜" 주입하기

모델은 **오늘이 며칠인지 모릅니다.** "내일", "다음주 월요일"을 정확한 날짜로 바꾸려면 기준 날짜를 반드시 알려줘야 합니다.
요일까지 함께 주면 "다음주 월요일" 같은 계산이 훨씬 정확해집니다.

In [76]:
WEEKDAYS = ["월", "화", "수", "목", "금", "토", "일"]
today_dt = datetime.strptime(TODAY, "%Y-%m-%d")

ROOM_SYSTEM_PROMPT = f"""당신은 사내 회의실 예약 비서입니다.
오늘 날짜: {TODAY} ({WEEKDAYS[today_dt.weekday()]}요일)

[규칙]
1. "내일", "다음주 월요일", "오후 3시" 같은 표현은 오늘 날짜 기준으로 YYYY-MM-DD, HH:MM(24시간)으로 변환하세요.
2. 종료 시간 언급이 없으면 1시간 회의로 간주하세요.
3. 예약 요청을 받으면 먼저 check_room_availability로 확인한 뒤 book_room을 호출하세요.
4. 날짜/시간 등 필수 정보가 불명확하면 추측하지 말고 사용자에게 되물어보세요.
5. 도구가 error를 반환하면 그 이유를 사용자에게 설명하고 대안을 제시하세요.
6. 답변은 간결하게, 회의실은 '이름(ID)' 형식으로 안내하세요."""

print(ROOM_SYSTEM_PROMPT)

당신은 사내 회의실 예약 비서입니다.
오늘 날짜: 2026-10-08 (목요일)

[규칙]
1. "내일", "다음주 월요일", "오후 3시" 같은 표현은 오늘 날짜 기준으로 YYYY-MM-DD, HH:MM(24시간)으로 변환하세요.
2. 종료 시간 언급이 없으면 1시간 회의로 간주하세요.
3. 예약 요청을 받으면 먼저 check_room_availability로 확인한 뒤 book_room을 호출하세요.
4. 날짜/시간 등 필수 정보가 불명확하면 추측하지 말고 사용자에게 되물어보세요.
5. 도구가 error를 반환하면 그 이유를 사용자에게 설명하고 대안을 제시하세요.
6. 답변은 간결하게, 회의실은 '이름(ID)' 형식으로 안내하세요.


## 2-4. 실험: 자연어 → 파라미터 추출 관찰

도구를 실행하지 않고 **모델이 어떤 인자를 추출하는지만** 확인해 봅니다.

In [78]:
extraction_tests = [
    "내일 오후 3시부터 4시까지 6명 들어갈 방 있어?",
    "다음주 월요일 오전 10시 반에 회의실 필요해요",          # 종료시간·인원 없음
    "다음 주 수요일 점심 먹고 2시부터 두 시간 정도, 15명이요",
    "회의실 좀 잡아줘",                                      # 정보 부족 → 되묻기 기대
]

for q in extraction_tests:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": ROOM_SYSTEM_PROMPT},
                  {"role": "user", "content": q}],
        tools=room_tools,
    )
    m = r.choices[0].message
    print("요청:", q)
    if m.tool_calls:
        for tc in m.tool_calls:
            print(f"  → {tc.function.name}: {tc.function.arguments}")
    else:
        print(f"  → (도구 호출 없음) {m.content}")
    print()

요청: 내일 오후 3시부터 4시까지 6명 들어갈 방 있어?
  → check_room_availability: {"date":"2026-10-09","start_time":"15:00","end_time":"16:00","min_capacity":6}

요청: 다음주 월요일 오전 10시 반에 회의실 필요해요
  → check_room_availability: {"date":"2026-10-16","start_time":"10:30","end_time":"11:30","min_capacity":null}

요청: 다음 주 수요일 점심 먹고 2시부터 두 시간 정도, 15명이요
  → check_room_availability: {"date":"2026-10-14","start_time":"14:00","end_time":"16:00","min_capacity":15}

요청: 회의실 좀 잡아줘
  → (도구 호출 없음) 회의실 예약을 위해 필요한 날짜, 시간, 그리고 인원 수를 알려주시면 감사하겠습니다.



**관찰 포인트**

- "오후 3시" → `15:00`, "10시 반" → `10:30` 처럼 형식이 정확히 변환됩니다.
- 인원 언급이 없으면 `min_capacity: null` → Strict 모드의 nullable 선택 인자가 동작합니다.
- 정보가 부족한 *"회의실 좀 잡아줘"* 는 도구를 호출하지 않고 **되묻습니다** (시스템 프롬프트 규칙 4).

## 2-5. 안전한 도구 실행기 만들기 — 에러를 "결과"로 돌려주기

시나리오 1에서는 `func(**args)`를 그대로 호출했습니다. 하지만 실무에서는 다음 문제가 생길 수 있습니다.

| 문제 | 그대로 두면 | 해결 |
|---|---|---|
| `arguments`가 깨진 JSON | `JSONDecodeError`로 프로그램 중단 | 파싱 실패를 에러 결과로 반환 |
| 등록되지 않은 함수 이름 | `KeyError` | 레지스트리 확인 후 에러 결과 반환 |
| 비즈니스 규칙 위반 (중복 예약 등) | `ValueError`로 중단 | **에러 메시지를 모델에 전달 → 모델이 대안 제시** |
| 예상치 못한 내부 오류 | 스택트레이스 노출 | 내부 정보는 숨기고 일반 메시지만 전달 |

> **핵심 패턴**: 도구 실행 실패 시 예외를 던지지 말고 `{"error": "이유"}`를 **tool 메시지로 돌려주세요.**
> 모델은 에러 내용을 읽고 *다른 시간을 제안*하거나 *사용자에게 되묻는* 등 스스로 복구합니다.

In [79]:
def execute_tool_call(tool_call, registry: dict, approval_fn=None, needs_approval: set = frozenset()) -> dict:
    """도구 호출 1건을 안전하게 실행하고, 항상 dict 결과를 반환한다 (예외를 밖으로 던지지 않음)."""
    name = tool_call.function.name

    # 1) 인자 파싱
    try:
        args = json.loads(tool_call.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"error": "도구 인자(JSON) 파싱에 실패했습니다. 올바른 형식으로 다시 호출하세요."}

    # 2) 허용된 도구인지 확인
    if name not in registry:
        return {"error": f"'{name}'은(는) 사용할 수 없는 도구입니다."}

    # 3) 쓰기 작업이면 사람 승인 받기
    if name in needs_approval and approval_fn is not None:
        if not approval_fn(name, args):
            return {"status": "rejected", "message": "사용자가 이 작업의 실행을 승인하지 않았습니다."}

    # 4) 실행 (비즈니스 에러는 모델에게 전달, 내부 에러는 숨김)
    try:
        return registry[name](**args)
    except (ValueError, TypeError) as e:
        return {"error": str(e)}
    except Exception as e:
        print(f"  [내부 오류 로그] {name}: {e!r}")   # 운영 로그에만 남김
        return {"error": "일시적인 시스템 오류가 발생했습니다. 잠시 후 다시 시도해 주세요."}

## 2-6. 사람 승인(Human-in-the-loop) 함수

`book_room`처럼 **실제 변경을 일으키는 도구**는 실행 직전에 사람에게 확인받습니다.
- `console_approval`: `input()`으로 직접 y/n 입력 (실제 서비스에서는 메신저의 [승인] 버튼 등으로 대체)
- `auto_approve` / `auto_reject`: 실습 편의를 위한 자동 승인/거절

In [ ]:
WRITE_TOOLS = {"book_room"}   # 승인이 필요한 쓰기 도구 목록


def console_approval(name: str, args: dict) -> bool:
    print("\n  ┌─ 승인 요청 ─────────────────────────────")
    print(f"  │ 작업: {name}")
    for k, v in args.items():
        print(f"  │ {k}: {v}")
    print("  └─────────────────────────────────────────")
    return input("  실행할까요? (y/n): ").strip().lower() == "y"


def auto_approve(name, args):
    print(f"  [자동 승인] {name} {args}")
    return True


def auto_reject(name, args):
    print(f"  [자동 거절] {name} {args}")
    return False

## 2-7. 멀티턴 대화가 가능한 예약 비서

실제 메신저처럼 **대화 기록(history)을 유지**하며 여러 번 주고받을 수 있게 만듭니다.

In [81]:
def room_chat(history: list, user_text: str, approval_fn=auto_approve, max_rounds: int = 6) -> str:
    history.append({"role": "user", "content": user_text})

    for _ in range(max_rounds):
        r = client.chat.completions.create(model=MODEL, messages=history, tools=room_tools)
        msg = r.choices[0].message

        if not msg.tool_calls:
            history.append({"role": "assistant", "content": msg.content})
            return msg.content

        history.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })
        print(f"  (이번 응답의 도구 호출 수: {len(msg.tool_calls)})")
        for tc in msg.tool_calls:
            print(f"  → {tc.function.name}({tc.function.arguments})")
            result = execute_tool_call(tc, ROOM_TOOL_REGISTRY, approval_fn, WRITE_TOOLS)
            history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "요청 처리 단계가 너무 많아 중단했습니다. 요청을 나눠서 다시 말씀해 주세요."


def new_room_session():
    """새 대화 세션 시작 + 예약 데이터 초기화"""
    RESERVATIONS[:] = copy.deepcopy(INITIAL_RESERVATIONS)
    return [{"role": "system", "content": ROOM_SYSTEM_PROMPT}]

In [82]:
# ---- 멀티턴 대화 시나리오 ----
history = new_room_session()

print("직원: 내일 오후 3시부터 4시까지 6명 회의할 방 있어?")
print("비서:", room_chat(history, "내일 오후 3시부터 4시까지 6명 회의할 방 있어?"), "\n")

print("직원: 그럼 거기로 '분기 실적 리뷰'로 예약해줘")
print("비서:", room_chat(history, "그럼 거기로 '분기 실적 리뷰'로 예약해줘"), "\n")

print("현재 예약 목록:")
for r in RESERVATIONS:
    print(" ", r)

직원: 내일 오후 3시부터 4시까지 6명 회의할 방 있어?
  (이번 응답의 도구 호출 수: 1)
  → check_room_availability({"date":"2026-10-09","start_time":"15:00","end_time":"16:00","min_capacity":6})
  (이번 응답의 도구 호출 수: 1)
  → book_room({"room_id":"R301","date":"2026-10-09","start_time":"15:00","end_time":"16:00","title":"회의"})
  [자동 승인] book_room {'room_id': 'R301', 'date': '2026-10-09', 'start_time': '15:00', 'end_time': '16:00', 'title': '회의'}
비서: 내일(2026-10-09) 오후 3시부터 4시까지 대회의실(R301)이 예약되었습니다. 성공적으로 예약되었습니다! 

직원: 그럼 거기로 '분기 실적 리뷰'로 예약해줘
  (이번 응답의 도구 호출 수: 1)
  → book_room({"room_id":"R301","date":"2026-10-09","start_time":"15:00","end_time":"16:00","title":"분기 실적 리뷰"})
  [자동 승인] book_room {'room_id': 'R301', 'date': '2026-10-09', 'start_time': '15:00', 'end_time': '16:00', 'title': '분기 실적 리뷰'}
비서: 대회의실(R301)은 이미 '회의'로 예약되어 있어 '분기 실적 리뷰'로 변경할 수 없습니다. 예약을 취소하고 다시 예약할 수 있지만, 현재 시간에는 방이 없으므로 다른 방이나 시간으로 고려해 보셔야 합니다. 추가 요청이 있으신가요? 

현재 예약 목록:
  {'id': 'RSV-001', 'room_id': 'R102', 'date': '2026-10-09', 'start_time': '14:0

**관찰 포인트**

- 내일(10/9) 15~16시에는 R102, R201이 이미 예약돼 있어 **대회의실(R301)만** 가능하다고 안내합니다.
- 두 번째 턴의 *"거기로"* 를 이전 대화 맥락에서 R301로 해석합니다 → **history 유지의 효과**
- `book_room`은 실행 직전 승인 함수를 거칩니다.

> 직접 승인을 체험하려면 아래 셀처럼 `approval_fn=console_approval`을 넘겨 보세요. 거절(n)하면 모델이 어떻게 답하는지도 확인해 보세요.

In [85]:
# 직접 y/n 입력으로 승인 체험 (입력창이 나타납니다)
history = new_room_session()
print("비서:", room_chat(history, "다음주 월요일 오전 11시에 4명, 소회의실 A로 '주간 스크럼' 예약해줘",
                        approval_fn=console_approval))

  (이번 응답의 도구 호출 수: 2)
  → check_room_availability({"date": "2026-10-16", "start_time": "11:00", "end_time": "12:00", "min_capacity": 4})
  → check_room_availability({"date": "2026-10-16", "start_time": "11:00", "end_time": "12:00", "min_capacity": null})
  (이번 응답의 도구 호출 수: 1)
  → book_room({"room_id":"R101","date":"2026-10-16","start_time":"11:00","end_time":"12:00","title":"주간 스크럼"})

  ┌─ 승인 요청 ─────────────────────────────
  │ 작업: book_room
  │ room_id: R101
  │ date: 2026-10-16
  │ start_time: 11:00
  │ end_time: 12:00
  │ title: 주간 스크럼
  └─────────────────────────────────────────
비서: 회의실 '소회의실 A' 예약을 시도했지만, 사용자의 승인이 필요해 예약이 거부되었습니다. 예약을 진행하려면 확인이 필요합니다. 예약을 진행해도 괜찮으신가요?


## 2-8. 병렬 도구 호출(Parallel Tool Calls)

서로 **독립적인 조회 여러 건**이 필요하면 모델은 한 번의 응답에 `tool_calls`를 여러 개 담아 보냅니다.
API 왕복 횟수가 줄어 **응답 속도와 비용**이 개선됩니다.

> **주의**: `tool_calls`가 N개면 `role="tool"` 메시지도 **반드시 N개**, 각각 올바른 `tool_call_id`로 돌려줘야 합니다.
> 하나라도 빠지면 다음 API 호출에서 400 에러가 발생합니다.

In [86]:
history = new_room_session()
q = "내일, 다음주 월요일, 다음주 화요일 각각 오후 2시~3시에 8명 이상 들어가는 회의실 있는지 한 번에 알려줘"
print("직원:", q)
print("비서:", room_chat(history, q))

직원: 내일, 다음주 월요일, 다음주 화요일 각각 오후 2시~3시에 8명 이상 들어가는 회의실 있는지 한 번에 알려줘
  (이번 응답의 도구 호출 수: 3)
  → check_room_availability({"date": "2026-10-09", "start_time": "14:00", "end_time": "15:00", "min_capacity": 8})
  → check_room_availability({"date": "2026-10-12", "start_time": "14:00", "end_time": "15:00", "min_capacity": 8})
  → check_room_availability({"date": "2026-10-13", "start_time": "14:00", "end_time": "15:00", "min_capacity": 8})
비서: 내일(2026-10-09)과 다음주 월요일(2026-10-12), 그리고 화요일(2026-10-13) 오후 2시부터 3시까지 예약 가능한 8명 이상의 회의실은 다음과 같습니다:

- **내일 (2026-10-09)**
  - 중회의실 (R201): 10명 수용, 2층, [빔프로젝터, 화상회의]
  - 대회의실 (R301): 20명 수용, 3층, [빔프로젝터, 화상회의, 마이크]

- **다음주 월요일 (2026-10-12)**
  - 중회의실 (R201): 10명 수용, 2층, [빔프로젝터, 화상회의]
  - 대회의실 (R301): 20명 수용, 3층, [빔프로젝터, 화상회의, 마이크]

- **다음주 화요일 (2026-10-13)**
  - 중회의실 (R201): 10명 수용, 2층, [빔프로젝터, 화상회의]
  - 대회의실 (R301): 20명 수용, 3층, [빔프로젝터, 화상회의, 마이크]

예약을 원하시는 회의실과 날짜를 알려주세요.


`(이번 응답의 도구 호출 수: 3)` 처럼 **한 번에 3건**의 조회가 요청된 것을 확인할 수 있습니다.

병렬 호출을 끄고 싶다면(예: 호출 순서가 중요한 경우) 요청에 `parallel_tool_calls=False`를 지정합니다.

## 2-9. 에러 복구 실험

함수가 `{"error": ...}`를 돌려줄 때 모델이 어떻게 대응하는지 확인합니다.

In [87]:
error_cases = [
    "내일 오후 3시부터 4시까지 중회의실(R201) '디자인 리뷰'로 바로 예약해줘",   # 이미 예약된 시간 → 대안 제시
    "어제 오후 2시에 회의실 하나 잡아줘. 4명이야",                               # 지난 날짜
    "이번주 토요일 오전 10시에 4명 회의실",                                       # 주말
]

for q in error_cases:
    history = new_room_session()
    print("=" * 70)
    print("직원:", q)
    print("비서:", room_chat(history, q))

직원: 내일 오후 3시부터 4시까지 중회의실(R201) '디자인 리뷰'로 바로 예약해줘
  (이번 응답의 도구 호출 수: 2)
  → check_room_availability({"date": "2026-10-09", "start_time": "15:00", "end_time": "16:00", "min_capacity": null})
  → check_room_availability({"date": "2026-10-09", "start_time": "15:00", "end_time": "16:00", "min_capacity": null})
비서: '디자인 리뷰' 회의를 위해 요청하신 중회의실(R201)이 예약 가능하지 않습니다. 대신 사용 가능한 회의실은 다음과 같습니다:

1. 소회의실 A (R101): 4명 수용, 1층, TV 장비
2. 대회의실 (R301): 20명 수용, 3층, 빔프로젝터, 화상회의, 마이크 장비

원하시는 회의실 중 하나를 예약해드릴까요?
직원: 어제 오후 2시에 회의실 하나 잡아줘. 4명이야
비서: 어제는 2026-10-07입니다. 해당 시간이 지났기 때문에 회의실을 예약할 수 없습니다. 다른 날짜나 시간을 말씀해 주시면 도와드리겠습니다!
직원: 이번주 토요일 오전 10시에 4명 회의실
  (이번 응답의 도구 호출 수: 1)
  → check_room_availability({"date":"2026-10-10","start_time":"10:00","end_time":"11:00","min_capacity":4})
비서: 죄송하지만, 2026년 10월 10일은 주말로 회의실 예약이 불가능합니다. 평일로 다른 날짜를 선택해 주실 수 있을까요?


**관찰 포인트**

- 프로그램이 멈추지 않고, 모델이 에러 이유를 **사람이 이해하기 쉬운 말로 설명**합니다.
- 중복 예약 건은 모델이 스스로 다른 회의실을 조회해 **대안을 제시**하기도 합니다.
- 주말·과거 날짜 같은 규칙은 **프롬프트가 아니라 함수 코드에서 검증**했다는 점이 중요합니다.
  모델은 실수할 수 있으므로 **최종 방어선은 항상 코드**여야 합니다.

## 2-10. 연습 문제

**[문제 1]** 예약 취소 도구 `cancel_reservation(reservation_id)`를 추가하세요. 쓰기 작업이므로 `WRITE_TOOLS`에도 등록해 승인을 받도록 합니다.

**[문제 2]** `check_room_availability`에 `required_equipment` 파라미터(예: `"화상회의"`, nullable)를 추가해
*"내일 10시에 화상회의 되는 방"* 요청을 처리해 보세요. Strict 모드 규칙(required 포함, null 허용)을 지켜야 합니다.

**[문제 3]** `parallel_tool_calls=False`로 2-8 질문을 다시 실행하고, API 호출 횟수가 어떻게 달라지는지 비교해 보세요.

# [문제1]
cancel_reservation(reservation_id) 추가 WRITE_TOOLS에도 등록해 승은 받아야함.

In [92]:
# 함수지정
def cancel_reservation(reservation_id:str):
    """예약된 회의실의 예약을 취소한다. 예약된 회의실을 예약 가능한 회의실로 변경한다. reservation_id의 예시 RSV-001. (쓰기도구)"""

    for r in RESERVATIONS:
        if r["id"] == reservation_id :
            RESERVATIONS[:] = [t for t in RESERVATIONS if t["id"] != reservation_id]
            return {"success": True, "message":f"{reservation_id}의 예약은 취소되었습니다.", "room": ROOMS[r["room_id"]]["name"],"date":r["date"],"time":f"{r['start_time']}~{r['end_time']}","title":r["title"]}
    raise ValueError(
        f"{reservation_id}는 존재하지 않는 예약 번호입니다. 예약된 ID: {[r['id'] for r in RESERVATIONS]}"
    )

#==========================================================================================


# 함수 단독 테스트
# new_room_session()
# print(cancel_reservation("RSV-001"))                       # 성공 딕셔너리
# print([r["id"] for r in RESERVATIONS])                     # ['RSV-002', 'RSV-003']
# cancel_reservation("RSV-999")                              # ValueError, 가능한 ID가 메시지에 보임

#========================================================================================
reservation_cancel_tool={
        "type": "function",
        "function": {
            "name": "cancel_reservation",
            "description": (
                "회의실 예약을 취소한다. 되돌리기 어려운 쓰기 작업이므로, 사용자가 특정 회의실 예약취소를 명확히 요청했을 때만 호출한다."
            ),
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "reservation_id":    {"type": "string", "description": "예약된 id. 형식은 RSV-로 시작. 예시 RSV-001"},
                },
                "required": ["reservation_id"],
                "additionalProperties": False,
            },
        },
    }
# tool 등록
if reservation_cancel_tool not in room_tools:
    room_tools.append(reservation_cancel_tool)
ROOM_TOOL_REGISTRY["cancel_reservation"]=cancel_reservation
WRITE_TOOLS.add("cancel_reservation")

# [문제2]

In [101]:
# 2-2 `check_room_availability`에 `required_equipment` 파라미터(예: `"화상회의"`, nullable)를 추가해
# *"내일 10시에 화상회의 되는 방"* 요청을 처리해 보세요. Strict 모드 규칙(required 포함, null 허용)을 지켜야 합니다.
def check_room_availability(date: str, start_time: str, end_time: str, min_capacity=None,required_equipment=None) -> dict:
    """특정 날짜·시간대에 예약 가능한 회의실 목록을 반환한다.회의에 필요한 장비가 있으면 그 장비가 있는 방들만 방의 목록을 반환한다."""
    _validate_date(date)
    _validate_time_range(start_time, end_time)

    available = []
    for room_id, room in ROOMS.items():
        if min_capacity and room["capacity"] < min_capacity:
            continue
        #required_equipment 파라미터 추가
        if required_equipment and required_equipment not in room["equipment"]:
            continue
        busy = any(
            r["room_id"] == room_id and r["date"] == date
            and _is_overlap(start_time, end_time, r["start_time"], r["end_time"])
            for r in RESERVATIONS
        )
        if not busy:
            available.append({"room_id": room_id, **room})

    return {"date": date, "time": f"{start_time}~{end_time}",
            "available_count": len(available), "available_rooms": available}
ROOM_TOOL_REGISTRY["check_room_availability"] = check_room_availability

#=====================================================================================
# room_tools의 check_room_availability의 세부 내용 수정
for tool in room_tools:
    if tool["function"]["name"] == "check_room_availability":
        # Strict 규칙 1: 선택 인자는 null 허용으로 표현
        tool["function"]["parameters"]["properties"]["required_equipment"] = {
            "type": ["string", "null"],
            "description": "필요한 장비 (예: 화상회의, 빔프로젝터, TV, 화이트보드, 마이크). 언급이 없으면 null",
        }
        # Strict 규칙 2: 모든 속성을 required에 나열 (이미 있으면 중복해서 넣지 않음)
        if "required_equipment" not in tool["function"]["parameters"]["required"]:
            tool["function"]["parameters"]["required"].append("required_equipment")
#====================================================================================
# 정의한 함수 테스트
print(check_room_availability(date="2026-10-09", start_time="10:00", end_time="11:00", required_equipment="화상회의"))   # R201, R301 기대
print('\n', '=' * 70)
print("문제 2-2 테스트")
history = new_room_session()
print("비서:", room_chat(history, "내일 오전 10시에 화상회의 되는 방 있어?"))
print('\n', '=' * 70)
history = new_room_session()
print("비서:", room_chat(history, "내일 오전 10시에 화상 회의 되는 방 있어?"))

{'date': '2026-10-09', 'time': '10:00~11:00', 'available_count': 2, 'available_rooms': [{'room_id': 'R201', 'name': '중회의실', 'capacity': 10, 'floor': 2, 'equipment': ['빔프로젝터', '화상회의']}, {'room_id': 'R301', 'name': '대회의실', 'capacity': 20, 'floor': 3, 'equipment': ['빔프로젝터', '화상회의', '마이크']}]}

문제 2-2 테스트
  (이번 응답의 도구 호출 수: 1)
  → check_room_availability({"date":"2026-10-09","start_time":"10:00","end_time":"11:00","min_capacity":null,"required_equipment":"화상회의"})
비서: 내일 오전 10시에 예약 가능한 화상회의실은 다음과 같습니다:

1. 중회의실(R201) - 수용 인원: 10명, 2층
2. 대회의실(R301) - 수용 인원: 20명, 3층

어느 방을 예약할까요?

  (이번 응답의 도구 호출 수: 1)
  → check_room_availability({"date":"2026-10-09","start_time":"10:00","end_time":"11:00","min_capacity":null,"required_equipment":"화상회의"})
비서: 내일(2026-10-09) 오전 10시에 예약 가능한 화상 회의실은 다음과 같습니다:

1. 중회의실(R201) - 수용 인원: 10명, 2층
2. 대회의실(R301) - 수용 인원: 20명, 3층

어떤 방을 예약하시겠습니까?


# [문제3]

In [102]:
# [문제3] parallel_tool_calls=False로 2-8 질문을 다시 실행하고 API 호출 횟수 비교
import time

def room_chat_compare(user_text, parallel):
    """parallel이 True면 병렬 호출 허용, False면 한 번에 하나씩만 호출한다. API 호출 횟수와 걸린 시간을 출력한다."""
    history = new_room_session()                      # 예약 초기화 + 새 대화 기록
    history.append({"role": "user", "content": user_text})
    api_calls = 0                                     # API 호출 횟수 (처음 0)
    start = time.time()                               # 시작 시간

    for round_no in range(1, 9):                      # 최대 8라운드 (1~8)
        r = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=room_tools,
            parallel_tool_calls=parallel,             # 이 한 줄이 비교의 핵심 (True/False)
        )
        api_calls = api_calls + 1                     # 호출할 때마다 1 증가
        msg = r.choices[0].message

        if not msg.tool_calls:                        # 도구 호출이 없으면 = 최종 답변
            break

        # 도구 요청을 대화 기록에 추가하고 실행 (room_chat과 같은 흐름)
        history.append({"role": "assistant", "content": msg.content,
                        "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            result = execute_tool_call(tc, ROOM_TOOL_REGISTRY, auto_approve, WRITE_TOOLS)
            history.append({"role": "tool", "tool_call_id": tc.id,
                            "content": json.dumps(result, ensure_ascii=False)})

    print(f"parallel_tool_calls={parallel} | API 호출 {api_calls}번 | {round(time.time() - start, 1)}초")
    print("답변:", msg.content)

# 2-8 질문
question = "내일, 다음주 월요일, 다음주 화요일 각각 오후 2시~3시에 8명 이상 들어가는 회의실 있는지 한 번에 알려줘"

print("[병렬 허용]")
room_chat_compare(question, True)
print('\n', '=' * 70)
print("[병렬 금지]")
room_chat_compare(question, False)

[병렬 허용]
parallel_tool_calls=True | API 호출 2번 | 4.7초
답변: 다음 회의실 예약 가능 정보는 다음과 같습니다:

1. **2026-10-09 (금요일) 오후 2시~3시**
   - **중회의실 (R201)**: 수용 인원 10명, 2층, 장비: 빔프로젝터, 화상회의
   - **대회의실 (R301)**: 수용 인원 20명, 3층, 장비: 빔프로젝터, 화상회의, 마이크

2. **2026-10-16 (월요일) 오후 2시~3시**
   - **중회의실 (R201)**: 수용 인원 10명, 2층, 장비: 빔프로젝터, 화상회의
   - **대회의실 (R301)**: 수용 인원 20명, 3층, 장비: 빔프로젝터, 화상회의, 마이크

3. **2026-10-17 (화요일)**: 주말로 인해 회의실 운영이 없습니다.

어떤 회의실을 예약하시겠습니까?

[병렬 금지]
parallel_tool_calls=False | API 호출 4번 | 6.1초
답변: 다음 일정에 대해 8명 이상 수용 가능한 회의실 정보입니다:

- **2026-10-09 (금요일) 오후 2시~3시**
  - 중회의실 (R201): 수용인원 10명
  - 대회의실 (R301): 수용인원 20명

- **2026-10-12 (월요일) 오후 2시~3시**
  - 중회의실 (R201): 수용인원 10명
  - 대회의실 (R301): 수용인원 20명

- **2026-10-13 (화요일) 오후 2시~3시**
  - 중회의실 (R201): 수용인원 10명
  - 대회의실 (R301): 수용인원 20명

예약을 원하시는 날짜와 회의실을 말씀해 주세요.


> API 호출 횟수가 예측처럼 병렬을 허용하지 않을때 병렬을 허용할 때보다 2배가 되었다.  
> 병렬을 허용한 경우 1번째 호출 후 도구 요청을 한 응답에 진행하고 실행해서 2번째 호출을 진행한뒤 모델이 최종 답변을 하므로 2번 호출이지만  
>  
> 병렬을 허용하지 않을 경우 호출 한번에 한 도구만 요청을 하기 때문에 응답 한번에 한번씩 도구 요청을  
> 하게 되고 총 4번의 API호출이 생겼다.

---
# 시나리오 3. 매출 분석 리포트 에이전트 — 멀티스텝 에이전트 루프

## 배경

영업기획팀은 매달 초 다음 작업을 반복합니다.

1. 지난달·이번달 지역별 매출 조회
2. 지역별 전월 대비 성장률 계산
3. 성장률 최고/최저 지역 분석
4. 영업팀장 메일 주소 확인 후 보고 메일 발송

이 업무를 *"8월 대비 9월 실적 정리해서 팀장님께 보고해줘"* 한 마디로 처리하는 **에이전트**를 만듭니다.
에이전트는 **이전 도구 결과를 보고 다음에 무엇을 할지 스스로 계획**하며, 여러 도구를 순서대로 엮어 목표를 달성합니다.

## 학습 포인트

1. **에이전트 루프**: `tool_calls`가 없을 때까지 반복 + 최대 반복 횟수 제한
2. **도구 체이닝**: 앞 도구의 결과가 뒤 도구의 입력이 되는 흐름
3. **계산은 도구에 위임**: LLM은 산술에 약하므로 정확한 계산은 코드로
4. **안전장치**: 메일 수신자 화이트리스트, 반복 횟수 제한
5. **실행 추적(Trace)과 토큰 사용량** 기록 → 디버깅·비용 관리

## 3-1. 백엔드 준비: 매출 DB, 계산기, 사내 주소록, 메일 발송

In [115]:
# ---- 지역별 월 매출 (단위: 만원) ----
SALES_DB = {
    "2026-07": {"서울": 11800, "부산": 8100, "대구": 5300, "광주": 4000, "대전": 3900},
    "2026-08": {"서울": 12500, "부산": 8300, "대구": 5100, "광주": 4200, "대전": 4100},
    "2026-09": {"서울": 13100, "부산": 9600, "대구": 4900, "광주": 5050, "대전": 4300},
}

# ---- 사내 주소록 ----
EMPLOYEE_DIRECTORY = {
    "영업팀장":   {"name": "김영호", "email": "yh.kim@campusmall.co.kr"},
    "마케팅팀장": {"name": "박소연", "email": "sy.park@campusmall.co.kr"},
    "대표이사":   {"name": "최정훈", "email": "ceo@campusmall.co.kr"},
}

ALLOWED_EMAIL_DOMAIN = "@campusmall.co.kr"   # 안전장치: 사내 도메인으로만 발송 허용
OUTBOX = []                                  # 발송된 메일 보관함 (실제 발송 대신)


def get_sales_data(month: str, region=None) -> dict:
    """월별 지역 매출 조회. region이 None이면 전체 지역."""
    if month not in SALES_DB:
        raise ValueError(f"{month} 데이터가 없습니다. 조회 가능 월: {list(SALES_DB)}")
    data = SALES_DB[month]
    if region:
        if region not in data:
            raise ValueError(f"'{region}' 지역이 없습니다. 가능한 지역: {list(data)}")
        data = {region: data[region]}
    return {"month": month, "unit": "만원", "sales": data, "total": sum(data.values())}


def calculate_growth_rate(label: str, previous: float, current: float) -> dict:
    """전기 대비 성장률(%) 계산."""
    if previous == 0:
        raise ValueError("이전 값이 0이면 성장률을 계산할 수 없습니다.")
    rate = (current - previous) / previous * 100
    return {"label": label, "previous": previous, "current": current,
            "diff": current - previous, "growth_rate_pct": round(rate, 2)}


def find_employee(keyword: str) -> dict:
    """직책 또는 이름으로 직원 이메일 조회."""
    for role, info in EMPLOYEE_DIRECTORY.items():
        if keyword in role or keyword in info["name"]:
            return {"found": True, "role": role, **info}
    return {"found": False, "message": f"'{keyword}'에 해당하는 직원이 없습니다.",
            "available_roles": list(EMPLOYEE_DIRECTORY)}


def send_email(to: str, subject: str, body: str) -> dict:
    """메일 발송 (쓰기 작업). 사내 도메인만 허용."""
    if not to.endswith(ALLOWED_EMAIL_DOMAIN):
        raise ValueError(f"보안 정책상 사내 메일({ALLOWED_EMAIL_DOMAIN})로만 발송할 수 있습니다. 요청 수신자: {to}")
    message_id = f"MSG-{len(OUTBOX) + 1:04d}"
    OUTBOX.append({"id": message_id, "to": to, "subject": subject, "body": body})
    return {"success": True, "message_id": message_id, "to": to}


pretty(get_sales_data("2026-09"))
pretty(calculate_growth_rate("광주", 4200, 5050))

{
  "month": "2026-09",
  "unit": "만원",
  "sales": {
    "서울": 13100,
    "부산": 9600,
    "대구": 4900,
    "광주": 5050,
    "대전": 4300
  },
  "total": 36950
}
{
  "label": "광주",
  "previous": 4200,
  "current": 5050,
  "diff": 850,
  "growth_rate_pct": 20.24
}


## 3-2. 도구 명세

`calculate_growth_rate`처럼 **계산 전용 도구**를 따로 두는 이유:
LLM은 `(5050-4200)/4200` 같은 계산을 *그럴듯하지만 틀리게* 답할 수 있습니다. 보고서 숫자는 틀리면 안 되므로 코드에 맡깁니다.

In [ ]:
agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_sales_data",
            "description": "월별 지역 매출(단위: 만원)을 조회한다. region을 null로 주면 전체 지역과 합계를 반환한다.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "month":  {"type": "string", "description": "조회 월, YYYY-MM 형식 (예: 2026-09)"},
                    "region": {"type": ["string", "null"], "description": "지역명 (서울, 부산, 대구, 광주, 대전). 전체는 null"},
                },
                "required": ["month", "region"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_growth_rate",
            "description": "이전 값 대비 현재 값의 성장률(%)과 증감액을 정확히 계산한다. 성장률이 필요하면 직접 계산하지 말고 반드시 이 도구를 사용한다. 여러 지역은 지역마다 각각 호출한다.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "label":    {"type": "string", "description": "계산 대상 이름 (예: 서울, 전체)"},
                    "previous": {"type": "number", "description": "이전 기간 값"},
                    "current":  {"type": "number", "description": "현재 기간 값"},
                },
                "required": ["label", "previous", "current"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_employee",
            "description": "직책(예: 영업팀장) 또는 이름으로 직원의 이름과 이메일 주소를 조회한다. 메일 발송 전 수신자 주소 확인에 사용한다.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {"keyword": {"type": "string", "description": "직책 또는 이름"}},
                "required": ["keyword"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "이메일을 발송한다. 수신자 주소는 반드시 find_employee로 확인한 주소를 사용한다.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "to":      {"type": "string", "description": "수신자 이메일 주소"},
                    "subject": {"type": "string", "description": "메일 제목"},
                    "body":    {"type": "string", "description": "메일 본문 (인사말, 핵심 요약, 지역별 표, 시사점 포함)"},
                },
                "required": ["to", "subject", "body"],
                "additionalProperties": False,
            },
        },
    },
]

AGENT_TOOL_REGISTRY = {
    "get_sales_data": get_sales_data,
    "calculate_growth_rate": calculate_growth_rate,
    "find_employee": find_employee,
    "send_email": send_email,
}

AGENT_SYSTEM_PROMPT = """당신은 캠퍼스몰 영업기획팀의 데이터 분석 에이전트입니다.
[작업 원칙]
1. 숫자는 반드시 get_sales_data로 조회한 값만 사용하고, 성장률 등 계산은 calculate_growth_rate 도구로 수행하세요.
2. 서로 독립적인 조회·계산은 한 번에 병렬로 요청해 단계를 줄이세요.
3. 메일 수신자는 find_employee로 확인한 주소만 사용하세요.
4. 도구가 error를 반환하면 원인을 파악해 수정하거나, 불가능하면 사용자에게 이유를 설명하세요.
5. 작업이 끝나면 수행한 내용을 3~5줄로 요약 보고하세요."""

# 아래 답변에서 모델이 년도를 2023년으로 고정해서 답변해서 날짜를 줌으로서 지정된 데이터를 사용할 수 있도록 유도.
AGENT_SYSTEM_PROMPT = AGENT_SYSTEM_PROMPT + f"\n오늘 날짜: {TODAY} ({WEEKDAYS[today_dt.weekday()]}요일)"

## 3-3. 에이전트 루프 구현

시나리오 1·2의 루프를 **실무형**으로 확장합니다.

```
while 반복횟수 < max_iterations:
    응답 = 모델 호출(messages, tools)
    토큰 사용량 누적, 추적 로그 기록
    if tool_calls 없음:  → 최종 답변 반환 (종료)
    각 tool_call 실행 (안전 실행기 사용) → 결과를 messages에 추가
최대 반복 도달 → 안전하게 중단
```

`max_iterations`는 모델이 같은 도구를 계속 부르는 **무한 루프와 비용 폭주를 막는 필수 안전장치**입니다.

In [121]:
import time


def run_agent(user_input: str,
              system_prompt: str = AGENT_SYSTEM_PROMPT,
              tools: list = agent_tools,
              registry: dict = AGENT_TOOL_REGISTRY,
              max_iterations: int = 10,
              verbose: bool = True) -> dict:
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}]
    trace = []                                  # 실행 추적 로그
    usage = {"prompt_tokens": 0, "completion_tokens": 0}
    started = time.time()

    for step in range(1, max_iterations + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        usage["prompt_tokens"] += response.usage.prompt_tokens
        usage["completion_tokens"] += response.usage.completion_tokens

        # ---- 종료 조건: 더 이상 도구 호출이 없음 ----
        if not msg.tool_calls:
            if verbose:
                print(f"[Step {step}] 최종 답변 생성 완료")
            return {"answer": msg.content, "steps": step, "trace": trace,
                    "usage": usage, "elapsed_sec": round(time.time() - started, 1)}

        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})

        if verbose:
            print(f"[Step {step}] 도구 호출 {len(msg.tool_calls)}건")
        for tc in msg.tool_calls:
            result = execute_tool_call(tc, registry)
            is_error = isinstance(result, dict) and "error" in result
            trace.append({"step": step, "tool": tc.function.name,
                          "args": json.loads(tc.function.arguments), "error": is_error, "result": result})
            if verbose:
                flag = "  [ERROR]" if is_error else ""
                print(f"    - {tc.function.name}({tc.function.arguments}){flag}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})

    # ---- 안전장치: 최대 반복 도달 ----
    if verbose:
        print(f"최대 반복 횟수({max_iterations}) 도달 → 중단")
    return {"answer": "작업 단계가 허용 범위를 초과하여 중단했습니다. 요청을 더 작게 나눠 주세요.",
            "steps": max_iterations, "trace": trace, "usage": usage,
            "elapsed_sec": round(time.time() - started, 1)}

## 3-4. 워밍업: 단순 조회

In [122]:
result = run_agent("9월 부산 매출 얼마였어?")
print("\n답변:", result["answer"])
print("토큰 사용량:", result["usage"])

[Step 1] 도구 호출 1건
    - get_sales_data({"month":"2026-09","region":"부산"})
[Step 2] 최종 답변 생성 완료

답변: 2026년 9월 부산의 매출은 9,600만원이었습니다.
토큰 사용량: {'prompt_tokens': 1073, 'completion_tokens': 43}


## 3-5. 본 실습: 복합 업무 자동화

이제 4단계 업무 전체를 한 문장으로 요청합니다. 에이전트가 **어떤 순서로 계획하고 실행하는지** 단계별 로그를 주의 깊게 보세요.

In [123]:
OUTBOX.clear()

task = ("8월 대비 9월 지역별 매출 성장률을 계산하고, 성장률이 가장 높은 지역과 가장 낮은 지역을 짚어서 "
        "영업팀장님께 보고 메일 보내줘. 전체 매출 성장률도 포함해줘.")

result = run_agent(task)

print("\n" + "=" * 70)
print("에이전트 최종 보고")
print("=" * 70)
print(result["answer"])
print(f"\n총 {result['steps']} 단계 / 소요 {result['elapsed_sec']}초 / 토큰 {result['usage']}")

[Step 1] 도구 호출 2건
    - get_sales_data({"month": "2026-08", "region": null})
    - get_sales_data({"month": "2026-09", "region": null})
[Step 2] 도구 호출 6건
    - calculate_growth_rate({"label": "서울", "previous": 12500, "current": 13100})
    - calculate_growth_rate({"label": "부산", "previous": 8300, "current": 9600})
    - calculate_growth_rate({"label": "대구", "previous": 5100, "current": 4900})
    - calculate_growth_rate({"label": "광주", "previous": 4200, "current": 5050})
    - calculate_growth_rate({"label": "대전", "previous": 4100, "current": 4300})
    - calculate_growth_rate({"label": "전체", "previous": 34200, "current": 36950})
[Step 3] 도구 호출 1건
    - find_employee({"keyword":"영업팀장"})
[Step 4] 도구 호출 1건
    - send_email({"to":"yh.kim@campusmall.co.kr","subject":"8월 대비 9월 지역별 매출 성장률 보고","body":"안녕하세요, 김영호 팀장님.\n\n2026년 9월 지역별 매출 성장률에 대한 분석 결과를 보고 드립니다.\n\n- **서울**: 4.8% (증감액: 600 만원)\n- **부산**: 15.66% (증감액: 1300 만원)\n- **대구**: -3.92% (증감액: -200 만원)\n- **광주**: 20.24% (증감액: 850 만원)\n- **

In [124]:
# 실제로 "발송된" 메일 확인
for mail in OUTBOX:
    print(f"[{mail['id']}] To: {mail['to']}")
    print(f"제목: {mail['subject']}")
    print("-" * 60)
    print(mail["body"])

[MSG-0001] To: yh.kim@campusmall.co.kr
제목: 8월 대비 9월 지역별 매출 성장률 보고
------------------------------------------------------------
안녕하세요, 김영호 팀장님.

2026년 9월 지역별 매출 성장률에 대한 분석 결과를 보고 드립니다.

- **서울**: 4.8% (증감액: 600 만원)
- **부산**: 15.66% (증감액: 1300 만원)
- **대구**: -3.92% (증감액: -200 만원)
- **광주**: 20.24% (증감액: 850 만원)
- **대전**: 4.88% (증감액: 200 만원)
- **전체**: 8.04% (증감액: 2750 만원)

**가장 높은 성장률**: 광주 (20.24%)  
**가장 낮은 성장률**: 대구 (-3.92%)

향후 전략 수립에 참고하시기 바랍니다.

감사합니다.


## 3-6. 실행 추적(Trace) 분석

에이전트가 무엇을 했는지 단계별로 정리해 봅니다. 실무에서는 이 로그를 DB나 모니터링 도구(LangSmith, Langfuse 등)에 저장해
**오동작 원인 분석, 비용 추적, 품질 개선**에 활용합니다.

In [125]:
print(f"{'Step':<6}{'도구':<24}{'인자':<55}{'에러'}")
print("-" * 95)
for t in result["trace"]:
    args = json.dumps(t["args"], ensure_ascii=False)
    args = args if len(args) <= 52 else args[:49] + "..."
    print(f"{t['step']:<6}{t['tool']:<24}{args:<55}{'O' if t['error'] else ''}")

# 정답 검증: 코드로 직접 계산한 값과 에이전트가 사용한 값 비교
print("\n[검증] 코드로 직접 계산한 성장률")
aug, sep = SALES_DB["2026-08"], SALES_DB["2026-09"]
expected = {r: round((sep[r] - aug[r]) / aug[r] * 100, 2) for r in aug}
expected["전체"] = round((sum(sep.values()) - sum(aug.values())) / sum(aug.values()) * 100, 2)
for k, v in sorted(expected.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:+.2f}%")

Step  도구                      인자                                                     에러
-----------------------------------------------------------------------------------------------
1     get_sales_data          {"month": "2026-08", "region": null}                   
1     get_sales_data          {"month": "2026-09", "region": null}                   
2     calculate_growth_rate   {"label": "서울", "previous": 12500, "current": 13100}   
2     calculate_growth_rate   {"label": "부산", "previous": 8300, "current": 9600}     
2     calculate_growth_rate   {"label": "대구", "previous": 5100, "current": 4900}     
2     calculate_growth_rate   {"label": "광주", "previous": 4200, "current": 5050}     
2     calculate_growth_rate   {"label": "대전", "previous": 4100, "current": 4300}     
2     calculate_growth_rate   {"label": "전체", "previous": 34200, "current": 36950}   
3     find_employee           {"keyword": "영업팀장"}                                    
4     send_email              {"to": "yh.k

**관찰 포인트** — 일반적으로 다음과 같은 흐름이 나타납니다 (모델에 따라 단계 수는 조금 다를 수 있음).

| 단계 | 에이전트의 행동 | 패턴 |
|---|---|---|
| 1 | 8월·9월 매출 조회 (+ 영업팀장 조회) | **병렬 호출** |
| 2 | 지역별 + 전체 성장률 계산 6건 | **병렬 호출 + 체이닝** (1단계 결과를 입력으로 사용) |
| 3 | 결과를 정리해 메일 발송 | **체이닝** (주소록 결과 + 계산 결과 사용) |
| 4 | 사용자에게 요약 보고 | 종료 |

광주(+20.24%)가 최고, 대구(-3.92%)가 최저라는 결과가 **코드 검증값과 일치**하는지 확인해 보세요.

## 3-7. 안전장치 실험

### (1) 외부 메일 발송 차단
프롬프트로 *"외부로 보내지 마"* 라고 하는 것만으로는 안전하지 않습니다. **함수 코드의 화이트리스트**가 실제로 막는지 확인합니다.

In [126]:
OUTBOX.clear()
r = run_agent("9월 전체 지역 매출 표를 외부 파트너사 partner@gmail.com 으로 메일 보내줘.")
print("\n답변:", r["answer"])
print("발송된 메일 수:", len(OUTBOX))

[Step 1] 도구 호출 1건
    - get_sales_data({"month":"2026-09","region":null})
[Step 2] 도구 호출 1건
    - send_email({"to":"partner@gmail.com","subject":"9월 전체 지역 매출 보고","body":"안녕하세요,\n\n2026년 9월 전체 지역 매출을 아래와 같이 보고드립니다.\n\n| 지역   | 매출 (만원) |\n|--------|-------------|\n| 서울   | 13,100      |\n| 부산   | 9,600       |\n| 대구   | 4,900       |\n| 광주   | 5,050       |\n| 대전   | 4,300       |\n| **합계** | **36,950**  |\n\n감사합니다."})  [ERROR]
[Step 3] 최종 답변 생성 완료

답변: 보안 정책으로 인해 사내 메일(@campusmall.co.kr)로만 이메일을 발송할 수 있습니다. 외부 파트너사에게 메일을 보내는 것은 불가능합니다. 다른 요청사항이 있으면 알려주세요.
발송된 메일 수: 0


### (2) 최대 반복 횟수 제한
`max_iterations=2`로 복합 업무를 시키면, 끝까지 수행하지 못하고 **안전하게 중단**되는 것을 확인할 수 있습니다.

In [127]:
OUTBOX.clear()
r = run_agent(task, max_iterations=2)
print("\n답변:", r["answer"])
print("발송된 메일 수:", len(OUTBOX))

[Step 1] 도구 호출 12건
    - get_sales_data({"month": "2026-08", "region": null})
    - get_sales_data({"month": "2026-09", "region": null})
    - get_sales_data({"month": "2026-08", "region": "서울"})
    - get_sales_data({"month": "2026-09", "region": "서울"})
    - get_sales_data({"month": "2026-08", "region": "부산"})
    - get_sales_data({"month": "2026-09", "region": "부산"})
    - get_sales_data({"month": "2026-08", "region": "대구"})
    - get_sales_data({"month": "2026-09", "region": "대구"})
    - get_sales_data({"month": "2026-08", "region": "광주"})
    - get_sales_data({"month": "2026-09", "region": "광주"})
    - get_sales_data({"month": "2026-08", "region": "대전"})
    - get_sales_data({"month": "2026-09", "region": "대전"})
[Step 2] 도구 호출 6건
    - calculate_growth_rate({"label": "서울", "previous": 12500, "current": 13100})
    - calculate_growth_rate({"label": "부산", "previous": 8300, "current": 9600})
    - calculate_growth_rate({"label": "대구", "previous": 5100, "current": 4900})
    - calcula

### (3) 존재하지 않는 데이터 요청
10월 데이터는 DB에 없습니다. 에이전트가 에러를 받고 어떻게 대응하는지 보세요.

In [128]:
r = run_agent("9월 대비 10월 서울 매출 성장률 알려줘")
print("\n답변:", r["answer"])

[Step 1] 도구 호출 2건
    - get_sales_data({"month": "2026-09", "region": "서울"})
    - get_sales_data({"month": "2026-10", "region": "서울"})  [ERROR]
[Step 2] 최종 답변 생성 완료

답변: 10월 서울 매출 데이터가 존재하지 않아 성장률을 계산할 수 없습니다. 현재 조회 가능한 데이터는 9월의 매출 13,100만 원입니다. 10월 매출 데이터가 업데이트되면 다시 요청해 주세요.


## 3-8. 연습 문제

**[문제 1]** `send_email`에도 시나리오 2의 **사람 승인**을 적용해 보세요.
힌트: `run_agent` 안의 `execute_tool_call(tc, registry)`를 `execute_tool_call(tc, registry, approval_fn, {"send_email"})`로 바꾸고, `run_agent`에 `approval_fn` 인자를 추가합니다.

**[문제 2]** 7·8·9월 3개월 추세를 분석하는 도구 `get_sales_trend(region, months)`를 추가하고,
*"최근 3개월 동안 꾸준히 성장한 지역만 골라줘"* 요청을 처리해 보세요.

**[문제 3]** `AGENT_SYSTEM_PROMPT`에서 원칙 2(병렬 요청)를 지우고 다시 실행해 보세요. 단계 수와 토큰 사용량이 어떻게 달라지나요?

# [문제1]

In [129]:
# [3-8 문제1] send_email에도 사람 승인 적용

# ① run_agent를 다시 정의
def run_agent(user_input: str,
              system_prompt: str = AGENT_SYSTEM_PROMPT,
              tools: list = agent_tools,
              registry: dict = AGENT_TOOL_REGISTRY,
              max_iterations: int = 10,
              verbose: bool = True,
              approval_fn=None) -> dict:                          # (변경) 승인 함수를 받는 자리 추가, 기본값 None
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}]
    trace = []
    usage = {"prompt_tokens": 0, "completion_tokens": 0}
    started = time.time()

    for step in range(1, max_iterations + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        usage["prompt_tokens"] += response.usage.prompt_tokens
        usage["completion_tokens"] += response.usage.completion_tokens

        if not msg.tool_calls:
            if verbose:
                print(f"[Step {step}] 최종 답변 생성 완료")
            return {"answer": msg.content, "steps": step, "trace": trace,
                    "usage": usage, "elapsed_sec": round(time.time() - started, 1)}

        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})

        if verbose:
            print(f"[Step {step}] 도구 호출 {len(msg.tool_calls)}건")
        for tc in msg.tool_calls:
            # 승인 함수와 "승인이 필요한 도구 이름 집합"을 같이 넘김
            result = execute_tool_call(tc, registry, approval_fn, {"send_email"})
            is_error = isinstance(result, dict) and "error" in result
            trace.append({"step": step, "tool": tc.function.name,
                          "args": json.loads(tc.function.arguments), "error": is_error, "result": result})
            if verbose:
                flag = "  [ERROR]" if is_error else ""
                print(f"    - {tc.function.name}({tc.function.arguments}){flag}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})

    if verbose:
        print(f"최대 반복 횟수({max_iterations}) 도달 → 중단")
    return {"answer": "작업 단계가 허용 범위를 초과하여 중단했습니다. 요청을 더 작게 나눠 주세요.",
            "steps": max_iterations, "trace": trace, "usage": usage,
            "elapsed_sec": round(time.time() - started, 1)}

print("run_agent 재설정 완료 사람승인 추가")
print('\n', '=' * 70)

# ② 테스트용
task_short = "8월 대비 9월 서울 매출 성장률을 계산해서 영업팀장님께 메일 보내줘."

# ③ 승인한 경우: 메일이 발송돼야 함
OUTBOX.clear()
r = run_agent(task_short, approval_fn=auto_approve)
print("승인 → 발송된 메일 수:", len(OUTBOX))               # 1 기대
print('\n', '=' * 70)

# ④ 거절한 경우: 메일이 발송되지 않아야 함 (반복이 길어지지 않게 max_iterations 사용)
OUTBOX.clear()
r = run_agent(task_short, approval_fn=auto_reject, max_iterations=6)
print("거절 → 발송된 메일 수:", len(OUTBOX))               # 0 기대
print("답변:", r["answer"])

run_agent 재설정 완료 사람승인 추가

[Step 1] 도구 호출 3건
    - get_sales_data({"month": "2026-08", "region": "서울"})
    - get_sales_data({"month": "2026-09", "region": "서울"})
    - find_employee({"keyword": "영업팀장"})
[Step 2] 도구 호출 1건
    - calculate_growth_rate({"label":"서울","previous":12500,"current":13100})
[Step 3] 도구 호출 1건
  [자동 승인] send_email {'to': 'yh.kim@campusmall.co.kr', 'subject': '서울 매출 성장률 보고', 'body': '안녕하세요, 김영호 영업팀장님.\n\n2026년 8월 대비 9월 서울 매출의 성장률을 보고드립니다.\n\n- **8월 매출**: 12,500 만원\n- **9월 매출**: 13,100 만원\n- **증감액**: 600 만원\n- **성장률**: 4.8%\n\n이번 상승은 긍정적인 신호로 보입니다. 추가적인 전략 논의가 필요할 것으로 생각됩니다.\n\n감사합니다.'}
    - send_email({"to":"yh.kim@campusmall.co.kr","subject":"서울 매출 성장률 보고","body":"안녕하세요, 김영호 영업팀장님.\n\n2026년 8월 대비 9월 서울 매출의 성장률을 보고드립니다.\n\n- **8월 매출**: 12,500 만원\n- **9월 매출**: 13,100 만원\n- **증감액**: 600 만원\n- **성장률**: 4.8%\n\n이번 상승은 긍정적인 신호로 보입니다. 추가적인 전략 논의가 필요할 것으로 생각됩니다.\n\n감사합니다."})
[Step 4] 최종 답변 생성 완료
승인 → 발송된 메일 수: 1

[Step 1] 도구 호출 3건
    - get_sales_data({"month": "2026-08",

# [문제2]

In [130]:
# [3-8 문제2] 최근 추세 도구

# 함수: 지역의 최근 3개월 매출을 모아서 "매달 늘었는지" 판단
def get_sales_trend(region: str, months: int) -> dict:
    """지역의 최근 N개월 매출 추세를 계산한다. 매달 전월보다 늘었으면 steady_growth가 True."""
    if region not in SALES_DB["2026-09"]:                       # 없는 지역이면 에러 (get_sales_data와 같은 방식)
        raise ValueError(f"'{region}' 지역이 없습니다. 가능한 지역: {list(SALES_DB['2026-09'])}")
    month_list = sorted(SALES_DB)[-months:]                     
    values = [SALES_DB[m][region] for m in month_list]          
    steady = True                                               
    for i in range(1, len(values)):                             
        if values[i] <= values[i - 1]:                          
            steady = False
    return {"region": region, "months": month_list, "sales": values, "steady_growth": steady}

# 
trend_tool = {
    "type": "function",
    "function": {
        "name": "get_sales_trend",
        "description": "지역의 최근 N개월 월별 매출과 꾸준한 성장 여부(steady_growth)를 조회한다. 여러 지역은 지역마다 각각 호출한다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "region": {"type": "string", "description": "지역명 (서울, 부산, 대구, 광주, 대전)"},
                "months": {"type": "integer", "description": "최근 몇 개월을 볼지 (예: 3)"},
            },
            "required": ["region", "months"],
            "additionalProperties": False,
        },
    },
}

# tool 등록
if trend_tool not in agent_tools:
    agent_tools.append(trend_tool)
AGENT_TOOL_REGISTRY["get_sales_trend"] = get_sales_trend

# 
print(get_sales_trend("서울", 3))                               # steady_growth True 기대
print('\n', '=' * 70)
r = run_agent("최근 3개월 동안 꾸준히 성장한 지역만 골라줘.")
print(r["answer"])                                              # 서울, 부산, 광주, 대전 기대 (대구 제외)

{'region': '서울', 'months': ['2026-07', '2026-08', '2026-09'], 'sales': [11800, 12500, 13100], 'steady_growth': True}

[Step 1] 도구 호출 5건
    - get_sales_trend({"region": "서울", "months": 3})
    - get_sales_trend({"region": "부산", "months": 3})
    - get_sales_trend({"region": "대구", "months": 3})
    - get_sales_trend({"region": "광주", "months": 3})
    - get_sales_trend({"region": "대전", "months": 3})
[Step 2] 최종 답변 생성 완료
최근 3개월 동안 꾸준히 성장한 지역은 다음과 같습니다:

1. 서울
2. 부산
3. 광주
4. 대전

이 지역들은 꾸준한 매출 성장을 기록했습니다. 추가적인 정보가 필요하시면 말씀해 주세요.


# [문제3]

In [134]:
# [3-8 문제3] 원칙 2(병렬 요청)를 지운 복사본 프롬프트와 원본을 비교

# 병렬요청 없는 프롬프트
AGENT_SYSTEM_PROMPT_COPY = """당신은 캠퍼스몰 영업기획팀의 데이터 분석 에이전트입니다.
[작업 원칙]
1. 숫자는 반드시 get_sales_data로 조회한 값만 사용하고, 성장률 등 계산은 calculate_growth_rate 도구로 수행하세요.
3. 메일 수신자는 find_employee로 확인한 주소만 사용하세요.
4. 도구가 error를 반환하면 원인을 파악해 수정하거나, 불가능하면 사용자에게 이유를 설명하세요.
5. 작업이 끝나면 수행한 내용을 3~5줄로 요약 보고하세요.""" + f"\n오늘 날짜: {TODAY} ({WEEKDAYS[today_dt.weekday()]}요일)"

# 확인
print("원본에 원칙 2 있음:", "병렬로 요청" in AGENT_SYSTEM_PROMPT)            # True 기대
print("복사본에 원칙 2 있음:", "병렬로 요청" in AGENT_SYSTEM_PROMPT_COPY)       # False 기대
print('\n', '=' * 70)

# ③ 같은 요청 (연도를 넣어서 모델이 엉뚱한 연도로 조회하지 않게 함)
task = ("2026년 8월 대비 9월 지역별 매출 성장률을 계산하고, 성장률이 가장 높은 지역과 가장 낮은 지역을 짚어서 "
        "영업팀장님께 보고 메일 보내줘. 전체 매출 성장률도 포함해줘.")

# ④ 원본 프롬프트로 실행
OUTBOX.clear()
r1 = run_agent(task, system_prompt=AGENT_SYSTEM_PROMPT, verbose=False)

# ⑤ 복사본 프롬프트로 실행
OUTBOX.clear()
r2 = run_agent(task, system_prompt=AGENT_SYSTEM_PROMPT_COPY, verbose=False)

# ⑥ 비교 출력
print("[원본: 원칙 2 있음] 단계 수:", r1["steps"], "| 입력 토큰:", r1["usage"]["prompt_tokens"],
      "| 출력 토큰:", r1["usage"]["completion_tokens"], "| 소요:", r1["elapsed_sec"], "초")
print(r1["answer"])
print('='*70)
print("[복사본: 원칙 2 삭제] 단계 수:", r2["steps"], "| 입력 토큰:", r2["usage"]["prompt_tokens"],
      "| 출력 토큰:", r2["usage"]["completion_tokens"], "| 소요:", r2["elapsed_sec"], "초")
print(r2["answer"])

원본에 원칙 2 있음: True
복사본에 원칙 2 있음: False

[원본: 원칙 2 있음] 단계 수: 5 | 입력 토큰: 7888 | 출력 토큰: 902 | 소요: 10.0 초
2026년 9월 지역별 매출 성장률을 계산하여 영업팀장님께 보고 메일을 보내드렸습니다. 전체 매출 성장률은 8.04%이며, 광주시가 가장 높은 성장률(20.24%)을 기록했습니다. 반면, 대구지역은 유일하게 매출이 감소(-3.92%)하였습니다. 메일 발송이 성공적으로 완료되었습니다.
[복사본: 원칙 2 삭제] 단계 수: 5 | 입력 토큰: 6654 | 출력 토큰: 719 | 소요: 8.3 초
2026년 8월 대비 9월 지역별 매출 성장률을 계산하여 영업팀장님께 보고 메일을 발송했습니다. 각 지역의 성장률은 서울 4.8%, 부산 15.66%, 대구 -3.92%, 광주 20.24%, 대전 4.88%로, 전체 매출 성장률은 8.04%입니다. 가장 높은 성장률은 광주에서, 가장 낮은 성장률은 대구에서 나타났습니다.


---
> 병렬 요청을 지운 프롬프트로 원본과 4번씩 실행해서 비교했을 때 예측과 달리 단계 수는 원본과 복사본 모두 매번 5단계로 같음
>
> 병렬 요청부분을 지워도 도구 명세의 description에 "여러 지역은 지역마다 각각 호출한다"는 내용이  있어서 모델이 독립적인 조회와 계산을 한 번에 요청해서 단계 수가 늘어나지 않았을 가능성 있음
---

---
# 마무리

## 핵심 정리

| 개념 | 핵심 내용 | 실습 |
|---|---|---|
| Tool Calling 사이클 | 모델은 **요청만**, 실행은 **우리 코드**. 결과는 `role="tool"` + `tool_call_id`로 반환 | 시나리오 1 |
| 도구 명세 | description이 도구 선택 정확도를 좌우. `enum`으로 값 범위 제한 | 시나리오 1 |
| `tool_choice` | `auto` / `none` / `required` / 특정 함수 강제 | 시나리오 1 |
| 파라미터 추출 | 상대 날짜 처리를 위해 **오늘 날짜 주입** 필수 | 시나리오 2 |
| Strict 모드 | `additionalProperties: false` + 전부 required + null 허용으로 선택 인자 표현 | 시나리오 2, 3 |
| 병렬 호출 | `tool_calls` N개 → tool 메시지 N개 모두 반환 | 시나리오 2, 3 |
| 에러 처리 | 예외 대신 `{"error": ...}` 반환 → 모델이 스스로 복구 | 시나리오 2, 3 |
| 사람 승인 | 쓰기 작업 실행 직전 승인 단계 | 시나리오 2 |
| 에이전트 루프 | `tool_calls`가 없을 때까지 반복 + `max_iterations` 제한 | 시나리오 3 |
| 관측 가능성 | 실행 추적·토큰 사용량 기록 | 시나리오 3 |

## 실무 적용 체크리스트

- [ ] 도구 함수는 LLM 없이도 **단독 테스트**가 가능한가?
- [ ] description에 **언제 쓰는지 / 언제 쓰지 말아야 하는지**가 적혀 있는가?
- [ ] 함수 이름 → 함수 매핑은 **화이트리스트(레지스트리)** 방식인가? (`eval` 금지)
- [ ] 입력 검증과 권한 체크는 **프롬프트가 아니라 코드**에서 하는가?
- [ ] 에러를 모델이 이해할 수 있는 **명확한 메시지**로 돌려주는가?
- [ ] 결제·삭제·발송 등 **되돌리기 어려운 작업**에 사람 승인이 있는가?
- [ ] 무한 루프 방지를 위한 **최대 반복 횟수**가 있는가?
- [ ] 도구 호출 로그와 토큰 사용량을 **기록**하고 있는가?
- [ ] 도구 수가 너무 많지 않은가? (많아질수록 선택 정확도 하락 → 역할별 에이전트 분리 고려)

